# 16 — Grouped benchmark แบบครบใน notebook การวิเคราะห์วิจัยและหลักฐานใช้งานจริง

**โปรโตคอลที่โค้ดทำอยู่:** ใช้ `data/gold_fixed_v1` และ `merged_group` เดิม เป็น benchmark ที่เคยตรวจระหว่างพัฒนา ไม่ใช่ untouched confirmatory test ผล `gold_ws` เป็นการวิเคราะห์คลิปที่เคยเห็น ห้ามใช้แทน grouped-heldout evidence

**แนวทางถัดไปตามที่ตกลง:** grouped development CV จากข้อมูลเดิม พร้อม offline augmentation เฉพาะ inner-train ตามส่วน 17 ของ notebook 00 ไม่ต้องสร้าง confirmatory test ใหม่ก่อน train ตัวรับ CV handoff และ bounded GPU smoke ทั้งสอง arms อยู่ในส่วน 14 และผ่านการรันแล้ว ส่วน full matched-step CV runner ยังรอ implement/อนุมัติ Metadata เหตุการณ์/ไซต์ใหม่เลื่อนไว้ตามเจ้าของงาน

โค้ดโครงการอยู่ใน notebook cells ไม่ import `scripts/`, ไม่ใช้ `%run` หรือ subprocess training scripts ใช้แพ็กเกจ PyTorch/torchvision/Ultralytics/NumPy/OpenCV/skimage และ COCO/ONNX ตามงานได้

## คำถามวิจัยและรายการงาน
1. ตรวจ metrics และเกณฑ์ annotation: บริเวณวัสดุ ไม่ใช่เม็ดรายเม็ด
2. เปรียบเทียบ 6 native-recipe arms บน validation/test ชุดเดียวกัน เปิดเผยความต่าง resolution/pretraining/tuning
3. ทำ controlled ablations ด้าน data/resolution/postprocessing และหลาย seeds แยก folds ออกจาก seeds
4. วิเคราะห์ PR, boundary, localization/classification/duplicate/missed errors, quality slices และความต่างฉาก
5. วัด paired group-bootstrap uncertainty; การทดสอบ ROI/background dependence ยังต้องเพิ่มก่อนอ้างผล
6. ประเมิน labeled events อิสระ, OOD/abstention, latency/export และ monitoring เมื่อมีข้อมูล/อุปกรณ์ เป็นงานต่อยอด ไม่ใช่ gate ก่อน development training

Training/inference/test evaluation/hardware actions ต้องเปิดแยกกัน Artifacts เดิมใช้ได้เมื่อผ่าน dataset/recipe/checkpoint provenance ถ้าขาด label/metadata/hardware ให้ระบุช่องว่าง ไม่สร้างคะแนนขึ้นเอง รายงานผล CV บนข้อมูลเดิมโดยไม่อ้าง generalization ไปไซต์ใหม่

อ้างอิง: COCO detection-eval; Cheng et al. Boundary IoU (CVPR 2021); Bolya et al. TIDE (ECCV 2020); Mitchell et al. Model Cards (2019); NIST AI RMF MEASURE ลิงก์และขอบเขตรายงานอยู่ส่วนท้าย

In [1]:
import os, json, time, hashlib, platform, random, itertools, importlib.metadata
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt

EXP = next((p for p in [Path.cwd(), Path.cwd()/'experiment', Path.home()/'Desktop/Aggregate_Project/experiment'] if (p/'data/gold/split_audit.csv').is_file()), None)
if EXP is None:
    raise FileNotFoundError('Set EXP to the experiment directory containing data/gold/split_audit.csv.')
DATA = EXP/'data/gold_fixed_v1'
CLS = ['gravel','sand']
SEEDS = [42, 123, 2026]
DO_TRAIN = False
DO_INFER = False
DO_EVAL_TEST = False
DO_BOOTSTRAP = False
DO_EXPORT = False
IMGSZ = 960
CONF_FLOOR = 0.001
MAX_DETS = 100
EVAL_VERSION = 'nb16-v3-box-mask-numeric-cache'
STAMP = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUT = EXP/'results/nb16_v2'/STAMP
OUT.mkdir(parents=True, exist_ok=False)
CACHE_ROOT = EXP/'results/nb16_v2/cache'
CACHE_ROOT.mkdir(parents=True, exist_ok=True)
MODEL_ROOT = EXP/'runs/nb16_v2'

def file_sha(path):
    h = hashlib.sha256()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(1<<20), b''): h.update(block)
    return h.hexdigest()

def write_json(path, data):
    Path(path).write_text(json.dumps(data, indent=2, allow_nan=False, default=lambda x: x.item() if isinstance(x,np.generic) else str(x)), encoding='utf-8')

def savefig(name):
    plt.gcf().savefig(OUT/(name+'.pdf'), bbox_inches='tight')
    plt.gcf().savefig(OUT/(name+'.png'), dpi=300, bbox_inches='tight')
    plt.show(); plt.close()

plt.rcParams.update({'figure.dpi':90,'font.size':10,'axes.spines.top':False,'axes.spines.right':False})
versions = {}
for pkg in ['numpy','pandas','opencv-python','matplotlib','torch','torchvision','ultralytics','scipy','scikit-image','pycocotools','onnx','onnxruntime']:
    try: versions[pkg] = importlib.metadata.version(pkg)
    except importlib.metadata.PackageNotFoundError: versions[pkg] = 'NOT INSTALLED'
write_json(OUT/'environment.json', {'python':platform.python_version(),'platform':platform.platform(),'packages':versions})
print('Protocol:', DATA, '| output:', OUT)
print('No training/inference/test evaluation enabled by default.')

Protocol: /home/supawich/Desktop/Aggregate_Project/experiment/data/gold_fixed_v1 | output: /home/supawich/Desktop/Aggregate_Project/experiment/results/nb16_v2/20261014T154044079887Z
No training/inference/test evaluation enabled by default.


## 1. Dataset identity, merged-group heldout และ leakage gates

ใช้ group mapping เดิม ไม่ใช้ UUID ที่ตัดสั้น ไม่สร้าง event/site metadata ขึ้นเอง ตรวจ image/label pairing, geometry, class IDs และการแยก exact duplicates/merged groups Identity รวม bytes ของภาพ **และ** label Checkpoints ใหม่ต้องฝึกบน identity นี้ Cache/checkpoint ของ `bench_ws` ไม่เข้ากับโปรโตคอลหลัก

In [2]:
def dataset_manifest(root):
    root = Path(root); rows=[]
    audit = pd.read_csv(root/'split_audit.csv')
    assert {'image','new_split','merged_group'}<=set(audit) and audit.image.is_unique
    assert audit[['image','new_split','merged_group']].notna().all().all()
    mapping = audit.set_index('image')
    for split in ('train','valid','test'):
        ips = sorted(p for p in (root/split/'images').iterdir() if p.suffix.lower() in {'.png','.jpg','.jpeg','.bmp'})
        expected = {p.stem for p in ips}; labels = {p.stem for p in (root/split/'labels').glob('*.txt')}
        assert ips and len(expected)==len(ips) and expected==labels, f'{split}: image/label mismatch or empty split'
        for ip in ips:
            assert ip.name in mapping.index and mapping.loc[ip.name,'new_split']==split, f'Split mapping differs: {ip.name}'
            im = cv2.imread(str(ip)); assert im is not None, f'Corrupt image: {ip}'
            lt = root/split/'labels'/(ip.stem+'.txt'); count=[0,0]
            for line in lt.read_text().splitlines():
                p=line.split(); assert len(p)>=7 and (len(p)-1)%2==0
                c=int(p[0]); xy=np.asarray(p[1:],float).reshape(-1,2)
                assert c in (0,1) and np.isfinite(xy).all() and ((xy>=0)&(xy<=1)).all()
                area=.5*abs(np.dot(xy[:,0],np.roll(xy[:,1],1))-np.dot(xy[:,1],np.roll(xy[:,0],1)))
                assert len(np.unique(xy,axis=0))>=3 and area>0
                count[c]+=1
            rows.append(dict(stem=ip.stem,image=ip.name,path=str(ip),split=split,group=str(mapping.loc[ip.name,'merged_group']),image_sha=file_sha(ip),label_sha=file_sha(lt),height=im.shape[0],width=im.shape[1],gravel=count[0],sand=count[1],decoded_sha=hashlib.sha256(im.tobytes()+str(im.shape).encode()).hexdigest()))
    frame=pd.DataFrame(rows); assert frame.stem.is_unique
    assert frame.groupby('group').split.nunique().max()==1, 'Merged-group leakage'
    assert frame.groupby('image_sha').split.nunique().max()==1, 'Exact duplicate crosses splits'
    assert frame.groupby('decoded_sha').split.nunique().max()==1, 'Same decoded pixels cross splits'
    identity=hashlib.sha256(frame.sort_values('stem')[['stem','split','group','image_sha','label_sha']].to_csv(index=False).encode()).hexdigest()
    return frame,identity

MANIFEST,DATA_ID=dataset_manifest(DATA)
GROUP_MAP=MANIFEST.set_index('stem').group.to_dict()
MANIFEST.to_csv(OUT/'dataset_manifest.csv',index=False)
print(MANIFEST.groupby('split').agg(images=('stem','size'),groups=('group','nunique'),gravel=('gravel','sum'),sand=('sand','sum')))
print('Dataset identity:',DATA_ID)
HANDOFF=None; QUALITY_PATH=None
handoffs=sorted((EXP/'results/nb00_audit').glob('*/handoff.json'),reverse=True)
nb00=json.loads((EXP/'00_EDA.ipynb').read_text())
nb00_source=hashlib.sha256(json.dumps([''.join(c['source']) for c in nb00['cells'] if c['cell_type']=='code'],ensure_ascii=False).encode()).hexdigest()
for path in handoffs:
    item=json.loads(path.read_text())
    if item.get('schema')!='nb00-to-nb16-v1' or item.get('dataset_id')!=DATA_ID or item.get('source_id')!=nb00_source: continue
    for field in ['manifest','quality','quality_reference']:
        artifact=path.parent/item[field]
        if artifact.resolve().parent!=path.parent.resolve() or file_sha(artifact)!=item[field+'_sha']: raise ValueError('Handoff artifact path/checksum mismatch')
    assert item['quality_threshold_source']=='train'
    HANDOFF=item; QUALITY_PATH=path.parent/item['quality']; break
if HANDOFF:
    write_json(OUT/'nb00_handoff_used.json',HANDOFF)
    print('Verified nb00 handoff:',QUALITY_PATH,'|',HANDOFF['geometry_status'],'| near-duplicate review:',HANDOFF['near_duplicate_review_status'])
else: print('GAP: run nb00 setup and authoritative final audit; training is blocked until a matching geometry-verified handoff exists.')

def load_gt(split_dir):
    result={}
    for ip in sorted((Path(split_dir)/'images').iterdir()):
        if ip.suffix.lower() not in {'.png','.jpg','.jpeg','.bmp'}: continue
        image=cv2.imread(str(ip)); assert image is not None
        h,w=image.shape[:2]; masks,classes=[],[]
        for line in (Path(split_dir)/'labels'/(ip.stem+'.txt')).read_text().splitlines():
            p=line.split(); xy=np.asarray(p[1:],float).reshape(-1,2)*[w,h]
            mask=np.zeros((h,w),np.uint8); cv2.fillPoly(mask,[xy.astype(np.int32)],1)
            assert mask.any(), f'Empty rasterized GT: {ip.name}'
            masks.append(mask.astype(bool)); classes.append(int(p[0]))
        result[ip.stem]=(ip,masks,classes)
    return result
GT_VALID=load_gt(DATA/'valid'); GT_TEST=load_gt(DATA/'test')
print('GT loaded; no model predictions evaluated.')

       images  groups  gravel  sand
split                              
test      105      12     189   159
train     753       7    1301  1550
valid     226       7     320   374
Dataset identity: 9bf2fab6eb34eb82dc7c4aad4b4de736c217e203c2f0f7cb2e36c6118705b8b9
Verified nb00 handoff: /home/supawich/Desktop/Aggregate_Project/experiment/results/nb00_audit/20261014T075628792047Z/final_quality_flags.csv | shapely validity checked | near-duplicate review: PENDING


GT loaded; no model predictions evaluated.


In [3]:
# ============================================================
# TEST SEAL — gold/test is NOT a confirmatory holdout
# ============================================================
# Evidence for contamination (from nb00 history): test images were visually
# inspected during cleaning/QA; test errors were examined in nb16; and the
# gold_v2 scene-balanced resplit was chosen AFTER observing test errors.
# Therefore any test number from this split is descriptive only.
TEST_CONTAMINATION_ACK = False          # must be set True deliberately to read test
TEST_MANIFEST = MANIFEST[MANIFEST.split=='test'].sort_values('stem')
TEST_SHA = hashlib.sha256(TEST_MANIFEST[['stem','image_sha','label_sha','group']].to_csv(index=False).encode()).hexdigest()
TEST_LOG = OUT/'test_access_log.jsonl'
TEST_SEAL = dict(
    test_sha=TEST_SHA, confirmatory=False,
    status='CONTAMINATED: visually inspected, error-analysed and used to motivate post-hoc resplit',
    images=len(TEST_MANIFEST), groups=int(TEST_MANIFEST.group.nunique()),
    largest_group_image_share=float(TEST_MANIFEST.group.value_counts(normalize=True).max()),
    usable_for='descriptive development comparison only; never model/threshold selection',
)
write_json(OUT/'test_seal.json', TEST_SEAL)
def log_test_access(event):
    if not TEST_CONTAMINATION_ACK:
        raise PermissionError('gold/test is contaminated and not a confirmatory holdout. Set TEST_CONTAMINATION_ACK=True only to produce descriptive development-test numbers.')
    with open(TEST_LOG,'a',encoding='utf-8') as f:
        f.write(json.dumps(dict(event=event,utc=datetime.now(timezone.utc).isoformat(),test_sha=TEST_SHA))+'\n')
print('Test seal:', TEST_SEAL['status']); print('Test sha:', TEST_SHA[:16], '| groups:', TEST_SEAL['groups'], '| largest group share:', round(TEST_SEAL['largest_group_image_share'],3))

# ============================================================
# PRIMARY EVIDENCE DESIGN — leave-group-out folds over train+valid
# ============================================================
# Test is excluded from all development folds. Folds are assigned by merged group
# so no group appears in both training and held-out fold; each fold must contain
# both classes. Training per fold is not implemented here (needs per-fold datasets).
DEV = MANIFEST[MANIFEST.split!='test']
dev_groups = DEV.groupby('group').agg(images=('stem','size'),gravel=('gravel','sum'),sand=('sand','sum'))
dev_groups['total'] = dev_groups.gravel + dev_groups.sand
K = min(4, len(dev_groups))
order = list(dev_groups[dev_groups.sand>0].sort_values(['sand','total'],ascending=False).index) + list(dev_groups[dev_groups.sand==0].sort_values('total',ascending=False).index)
fold_load = np.zeros(K); fold_sand = np.zeros(K); FOLD_OF = {}
for g in order:
    row = dev_groups.loc[g]
    k = min(range(K), key=lambda i: (fold_sand[i] if row.sand>0 else 0, fold_load[i]))
    FOLD_OF[g] = k; fold_load[k] += row.total; fold_sand[k] += row.sand>0
FOLD_TABLE = dev_groups.assign(fold=pd.Series(FOLD_OF)).reset_index()
FOLD_SUMMARY = FOLD_TABLE.groupby('fold').agg(groups=('group','nunique'),images=('images','sum'),gravel=('gravel','sum'),sand=('sand','sum'))
assert (FOLD_SUMMARY.gravel>0).all() and (FOLD_SUMMARY.sand>0).all(), 'Each development fold must contain both classes'
FOLD_TABLE.to_csv(OUT/'dev_group_folds.csv', index=False)
print(FOLD_SUMMARY.to_string())
print('Development groups:', len(dev_groups), '| folds:', K, '| test excluded from folds')


Test seal: CONTAMINATED: visually inspected, error-analysed and used to motivate post-hoc resplit
Test sha: 5cf27d13cfecd436 | groups: 12 | largest group share: 0.533
      groups  images  gravel  sand
fold                              
0          1     397     225  1396
1          3     220     466   187
2          4     199     490   187
3          6     163     440   154
Development groups: 14 | folds: 4 | test excluded from folds


## 2. ตรวจ evaluator — COCO 101-point AP และการจับคู่แบบหนึ่งต่อหนึ่ง

Inline reference รองรับ annotations ที่ไม่ใช่ crowd/ignore ของชุดนี้ ใช้ `pycocotools.COCOeval` สำหรับคะแนนหลัก AP/AP50/AP75/APs/APm/APl/AR เมื่อมีแพ็กเกจ ถ้าไม่มีต้องระบุชัด Reference matching เรียง score แล้วเลือก GT ที่ยังว่างและ IoU สูงสุด ไม่ลากสามเหลี่ยมต่อ recall=1 คลาสไม่มี GT เป็น N/A Size recall ใช้คู่เดียวกัน Threshold เลือกจาก validation **เท่านั้น**

Boundary IoU ใช้ inner boundary bands จาก erosion ทั้งสองฝั่ง โดยปรับตามเส้นทแยงมุมภาพ Error taxonomy ด้านล่างเป็นการบรรยาย ไม่ใช่ TIDE oracle-dAP ซึ่งต้องใช้ canonical analysis แยก

In [4]:
def as_box(value):
    a = np.asarray(value)
    if a.ndim == 1:
        if a.shape != (4,) or not np.isfinite(a).all() or (a[2:] <= a[:2]).any():
            raise ValueError('Expected finite positive-area xyxy box')
        return a.astype(float)
    if a.ndim != 2 or not a.any():
        raise ValueError('Expected nonempty HxW mask')
    ys,xs = np.where(a.astype(bool))
    return np.array([xs.min(),ys.min(),xs.max()+1,ys.max()+1],float)

def iou_mat(pm,gm):
    out = np.zeros((len(pm),len(gm)),float)
    for i,p in enumerate(pm):
        for j,g in enumerate(gm):
            if np.asarray(p).ndim == 1 or np.asarray(g).ndim == 1:
                a,b = as_box(p),as_box(g)
                inter = np.maximum(0,np.minimum(a[2:],b[2:])-np.maximum(a[:2],b[:2])).prod()
                union = (a[2:]-a[:2]).prod()+(b[2:]-b[:2]).prod()-inter
            else:
                a,b = np.asarray(p,bool),np.asarray(g,bool)
                if a.shape != b.shape: raise ValueError('Prediction/GT shape mismatch')
                inter = (a&b).sum(); union = (a|b).sum()
            out[i,j] = inter/union if union else 0.
    return out

def validate_predictions(preds,gt,task=None):
    if set(preds) != set(gt): raise ValueError('Prediction IDs must exactly cover the evaluation cohort, including empty results')
    for st,pl in preds.items():
        ip,gm,gc = gt[st]
        if len(gm) != len(gc): raise ValueError('GT mask/class length mismatch')
        shape = gm[0].shape if gm else cv2.imread(str(ip)).shape[:2]
        for c,s,m in pl:
            if c not in (0,1) or int(c) != c or not np.isfinite(s) or not 0 <= s <= 1:
                raise ValueError('Invalid prediction class/confidence')
            a = np.asarray(m)
            if a.ndim == 1:
                box = as_box(a)
                if task == 'segment' or (box[:2]<0).any() or box[2]>shape[1] or box[3]>shape[0]:
                    raise ValueError('Wrong representation or out-of-image box')
            elif a.ndim != 2 or a.shape != shape or not np.isfinite(a).all() or not a.any() or not np.isin(a,[0,1]).all() or task == 'detect':
                raise ValueError('Wrong representation or invalid binary mask')

def match_image(pl,gm,gc,threshold=.5,conf=0.,class_aware=True,max_dets=MAX_DETS):
    ordered = sorted(enumerate(pl),key=lambda x:-float(x[1][1]))
    used,counts,records = set(),{},[]
    for pi,(c,s,m) in ordered:
        if c not in (0,1) or not np.isfinite(s) or not 0<=s<=1: raise ValueError('Invalid prediction class/score')
        if s < conf: continue
        counts[c] = counts.get(c,0)+1
        if counts[c] > max_dets: continue
        candidates = [j for j in range(len(gm)) if j not in used and (not class_aware or gc[j]==c)]
        overlaps = iou_mat([m],[gm[j] for j in candidates])[0] if candidates else np.array([])
        j = candidates[int(overlaps.argmax())] if len(overlaps) and overlaps.max() >= threshold else None
        if j is not None: used.add(j)
        records.append(dict(pred=pi,cls=int(c),score=float(s),gt=j,tp=int(j is not None),iou=float(overlaps.max()) if len(overlaps) else 0.))
    return records,used

def pr_curve(preds,gt,c,iou=.5):
    if set(preds)-set(gt): raise ValueError('Unknown prediction image IDs')
    n = sum(sum(x==c for x in gc) for _,gm,gc in gt.values())
    records = []
    for st,(_,gm,gc) in gt.items():
        matched,_ = match_image(preds.get(st,[]),gm,gc,iou)
        records.extend(r for r in matched if r['cls']==c)
    records.sort(key=lambda r:-r['score'])
    tp = np.cumsum([r['tp'] for r in records],dtype=float)
    precision = tp/np.arange(1,len(tp)+1) if len(tp) else np.array([])
    recall = tp/n if n else np.zeros(len(tp))
    envelope = np.maximum.accumulate(precision[::-1])[::-1] if len(tp) else precision
    samples = np.zeros(101)
    for i,r in enumerate(np.linspace(0,1,101)):
        ids = np.flatnonzero(recall >= r)
        if len(ids): samples[i] = envelope[ids[0]]
    return dict(gt=n,ap=float(samples.mean()) if n else None,precision=precision,recall=recall,scores=np.array([r['score'] for r in records]),samples=samples)

def eval_reference(preds,gt):
    out = {}
    for c,cn in enumerate(CLS):
        aps = [pr_curve(preds,gt,c,t)['ap'] for t in np.linspace(.5,.95,10)]
        out[cn] = {'AP':float(np.mean(aps)) if aps[0] is not None else None,'AP50':aps[0],'AP75':aps[5]}
    active = [v['AP'] for v in out.values() if v['AP'] is not None]
    out['AP_macro'] = float(np.mean(active)) if active else None
    return out

def operating_metrics(preds,gt,thresholds):
    rows = []
    for c,cn in enumerate(CLS):
        tp=fp=fn=0; size={b:[0,0] for b in ['small','medium','large']}
        for st,(_,gm,gc) in gt.items():
            pl = [p for p in preds.get(st,[]) if p[0]==c]
            rr,used = match_image(pl,gm,gc,.5,float(thresholds[c]))
            tp += len(used); fp += len(rr)-len(used)
            ids = [j for j,x in enumerate(gc) if x==c]; fn += len(ids)-len(used)
            for j in ids:
                a = np.asarray(gm[j],bool).sum(); b='small' if a<1024 else ('medium' if a<9216 else 'large')
                size[b][1] += 1; size[b][0] += int(j in used)
        rows.append(dict(cls=cn,threshold=float(thresholds[c]),tp=tp,fp=fp,fn=fn,precision=tp/(tp+fp) if tp+fp else None,recall=tp/(tp+fn) if tp+fn else None,F1=2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else None,**{b+'_n':v[1] for b,v in size.items()},**{b+'_recall':v[0]/v[1] if v[1] else None for b,v in size.items()}))
    return pd.DataFrame(rows)

def select_thresholds(preds,valid):
    thresholds = {}
    for c in (0,1):
        curve = pr_curve(preds,valid,c)
        if not curve['gt']: raise ValueError('Validation must contain both classes for threshold selection')
        if not len(curve['scores']): thresholds[c]=float(np.nextafter(1.,2.)); continue
        scores = curve['scores']; last = np.r_[np.diff(scores)!=0,True]
        f = 2*curve['precision']*curve['recall']/np.maximum(curve['precision']+curve['recall'],1e-12)
        ids = np.flatnonzero(last); thresholds[c] = float(scores[ids[np.argmax(f[ids])]])
    return thresholds

def boundary_iou(p,g,ratio=.02):
    p,g=np.asarray(p,bool),np.asarray(g,bool)
    if p.shape!=g.shape: raise ValueError('Boundary mask shape mismatch')
    d=max(1,int(round(ratio*np.hypot(*g.shape))))
    def band(mask):
        padded=np.pad(mask.astype(np.uint8),d)
        eroded=cv2.erode(padded,np.ones((3,3),np.uint8),iterations=d)[d:-d,d:-d].astype(bool)
        return mask & ~eroded
    a,b=band(p),band(g); u=(a|b).sum()
    return float((a&b).sum()/u) if u else 0.

In [5]:
def canonical_coco(preds,gt,mode='segm'):
    if mode not in ('segm','bbox'): raise ValueError('Unsupported COCO mode')
    validate_predictions(preds,gt,'segment' if mode=='segm' else None)
    try:
        from pycocotools.coco import COCO
        from pycocotools.cocoeval import COCOeval
        from pycocotools import mask as mask_utils
    except ImportError:
        return {'status':'BLOCKED: pycocotools missing; reference metrics are diagnostic only'}
    import contextlib, io
    coco = COCO(); images=[]; anns=[]; detections=[]; aid=1
    for iid,(st,(ip,gm,gc)) in enumerate(gt.items(),1):
        shape=gm[0].shape if gm else cv2.imread(str(ip)).shape[:2]
        images.append(dict(id=iid,height=int(shape[0]),width=int(shape[1])))
        for m,c in zip(gm,gc):
            rle=mask_utils.encode(np.asfortranarray(np.asarray(m,np.uint8)))
            anns.append(dict(id=aid,image_id=iid,category_id=int(c)+1,segmentation=rle,area=float(mask_utils.area(rle)),bbox=mask_utils.toBbox(rle).tolist(),iscrowd=0)); aid+=1
        for c,s,m in preds[st]:
            item=dict(image_id=iid,category_id=int(c)+1,score=float(s))
            if mode=='segm': item['segmentation']=mask_utils.encode(np.asfortranarray(np.asarray(m,np.uint8)))
            else:
                box=as_box(m); item['bbox']=[float(box[0]),float(box[1]),float(box[2]-box[0]),float(box[3]-box[1])]
            detections.append(item)
    coco.dataset=dict(info={},images=images,annotations=anns,categories=[dict(id=i+1,name=c) for i,c in enumerate(CLS)])
    with contextlib.redirect_stdout(io.StringIO()):
        coco.createIndex()
        if detections: dt=coco.loadRes(detections)
        else:
            dt=COCO(); dt.dataset=dict(info={},images=images,annotations=[],categories=coco.dataset['categories']); dt.createIndex()
        ev=COCOeval(coco,dt,mode); ev.params.maxDets=[1,10,MAX_DETS]
        ev.evaluate(); ev.accumulate(); ev.summarize()
    names=['AP','AP50','AP75','AP_small','AP_medium','AP_large','AR1','AR10','AR100','AR_small','AR_medium','AR_large']
    result={k:float(v) if v>=0 else None for k,v in zip(names,ev.stats)}
    result['status']='canonical COCOeval'; result['mode']=mode; result['per_class']={}
    for i,cn in enumerate(CLS):
        p=ev.eval['precision'][:,:,i,0,-1]; values=p[p>=0]
        result['per_class'][cn]=float(values.mean()) if len(values) else None
    return result

im=Path('synthetic')
a=np.zeros((12,12),bool); a[:,:8]=1
b=np.zeros((12,12),bool); b[:,4:]=1
u=a|b
gt={'x':(im,[a,b],[0,0])}
assert len(match_image([(0,.9,a),(0,.8,u)],[a,b],[0,0])[1])==2
assert operating_metrics({'x':[(0,.9,u)]},gt,{0:0,1:0}).iloc[0].small_recall==.5
gt10={'x':(im,[a.copy() for _ in range(10)],[0]*10)}
assert np.isclose(pr_curve({'x':[(0,.9,a)]},gt10,0)['ap'],11/101)
assert pr_curve({},gt10,0)['ap']==0 and pr_curve({},gt10,1)['ap'] is None
assert np.isclose(boundary_iou(a,a),1)
assert match_image([(1,.9,a)],[a],[0])[1]==set()
assert match_image([(0,.9,a),(0,.8,a)],[a],[0])[1]=={0}
assert np.isclose(pr_curve({'x':[(0,.9,a),(0,.8,a)]},{'x':(im,[a],[0])},0)['ap'],1)
assert np.isclose(iou_mat([as_box(a)],[a])[0,0],1)
assert iou_mat([np.array([9,0,12,12])],[a])[0,0]==0
validate_predictions({'x':[(0,.9,as_box(a))]},gt10,'detect')
for bad in ({}, {'x':[(0,np.nan,a)]}, {'x':[(0,.5,np.zeros_like(a))]}):
    try: validate_predictions(bad,gt10)
    except ValueError: pass
    else: raise AssertionError('Invalid prediction contract accepted')
for mode,preds in [('segm',{'x':[(0,.9,a)]}),('bbox',{'x':[(0,.9,as_box(a))]})]:
    official=canonical_coco(preds,gt10,mode)
    if official['status']=='canonical COCOeval':
        assert np.isclose(official['AP'],eval_reference(preds,gt10)['AP_macro'],atol=1e-8)
    print(mode,official['status'])
print('Evaluator regression assertions passed; canonical parity is checked only when COCOeval is installed.')

segm canonical COCOeval
bbox canonical COCOeval
Evaluator regression assertions passed; canonical parity is checked only when COCOeval is installed.


## 3. หก arms สูตรฝึกเดิมของแต่ละโมเดล การเลือกด้วย validation และ provenance

A/B: pretrained YOLO26n/s segmentation; C: COCO-pretrained Mask R-CNN R50-FPNv2 พร้อมหัวใหม่ 3 คลาส; D: U-Net เริ่มสุ่ม + watershed; E/F: pretrained YOLO26n/s detection ใช้ validation เต็มชุดเดียวกัน เลือก checkpoint ด้วย canonical COCO AP เลือก confidence threshold จาก validation และใช้ test IDs เดียวกัน ความต่าง resolution/optimizer เป็น confounders ต้องเปิดเผย ไม่ใช่หลักฐานว่า architecture ดีกว่าโดยลำพัง

PyTorch loaders ใช้ `num_workers=0` เพื่อรันใน notebook ได้ ใช้ FP32 และ deterministic algorithms สำหรับ ROCm รัน jobs ทีละตัวไม่แย่ง GPU ต้องเปิด `DO_TRAIN` เอง สร้าง run directories ใหม่เพื่อไม่หยิบ checkpoint seen-scene เก่ามาใช้ YOLO เก็บทุก epoch แล้วเลือกด้วย canonical validation AP หลัง train ไม่ใช้ native fitness

ส่วนนี้ยังเป็น fixed-split benchmark ไม่ใช่ CV runner หรือ bounded smoke ของโปรโตคอลใหม่

In [6]:
RECIPES = [
    dict(name='yolo26n_seg',kind='yolo',task='segment',init='yolo26n-seg.pt',size=960,epochs=100,batch=8,lr=.001),
    dict(name='yolo26s_seg',kind='yolo',task='segment',init='yolo26s-seg.pt',size=960,epochs=100,batch=8,lr=.001),
    dict(name='mrcnn',kind='mrcnn',task='segment',init='COCO_DEFAULT',size=960,epochs=12,batch=2,lr=.005),
    dict(name='unet_ws',kind='unet',task='segment',init='random',size=512,epochs=40,batch=8,lr=.001),
    dict(name='yolo26n_det',kind='yolo',task='detect',init='yolo26n.pt',size=960,epochs=100,batch=8,lr=.001),
    dict(name='yolo26s_det',kind='yolo',task='detect',init='yolo26s.pt',size=960,epochs=100,batch=8,lr=.001),
]
write_json(OUT/'recipes.json',RECIPES)

nb_source = json.loads((EXP/'16_train_last.ipynb').read_text())
CODE_ID = hashlib.sha256(json.dumps([''.join(c.get('source',[])) for c in nb_source['cells'] if c['cell_type']=='code'],ensure_ascii=False).encode()).hexdigest()

def runtime():
    import torch
    if not hasattr(torch,'cuda'):
        raise RuntimeError('This kernel has an incomplete torch installation. Select the training environment; no automatic environment changes.')
    return torch, 'cuda:0' if torch.cuda.is_available() else 'cpu'

def seed_everything(seed):
    torch,device=runtime()
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True)
    if hasattr(torch.backends,'cudnn'):
        torch.backends.cudnn.benchmark=False; torch.backends.cudnn.deterministic=True
    return torch,device

def make_unet():
    torch,_=runtime(); nn=torch.nn
    def conv(i,o): return nn.Sequential(nn.Conv2d(i,o,3,padding=1),nn.ReLU(),nn.Conv2d(o,o,3,padding=1),nn.ReLU())
    class UNet(nn.Module):
        def __init__(self):
            super().__init__(); b=32
            self.e1,self.e2,self.e3,self.e4=conv(3,b),conv(b,b*2),conv(b*2,b*4),conv(b*4,b*8)
            self.mid=conv(b*8,b*16); self.pool=nn.MaxPool2d(2)
            self.ups=nn.ModuleList([nn.ConvTranspose2d(b*16,b*8,2,2),nn.ConvTranspose2d(b*8,b*4,2,2),nn.ConvTranspose2d(b*4,b*2,2,2),nn.ConvTranspose2d(b*2,b,2,2)])
            self.decs=nn.ModuleList([conv(b*16,b*8),conv(b*8,b*4),conv(b*4,b*2),conv(b*2,b)])
            self.head=nn.Conv2d(b,3,1)
        def forward(self,x):
            e1=self.e1(x); e2=self.e2(self.pool(e1)); e3=self.e3(self.pool(e2)); e4=self.e4(self.pool(e3))
            x=self.mid(self.pool(e4))
            for up,dec,skip in zip(self.ups,self.decs,[e4,e3,e2,e1]): x=dec(torch.cat([up(x),skip],1))
            return self.head(x)
    return UNet()

def make_mrcnn(pretrained=True):
    from torchvision.models.detection import maskrcnn_resnet50_fpn_v2
    from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
    from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor
    model=maskrcnn_resnet50_fpn_v2(weights='DEFAULT' if pretrained else None,weights_backbone=None,min_size=640,max_size=960,box_score_thresh=CONF_FLOOR,box_detections_per_img=MAX_DETS)
    model.roi_heads.box_predictor=FastRCNNPredictor(model.roi_heads.box_predictor.cls_score.in_features,3)
    model.roi_heads.mask_predictor=MaskRCNNPredictor(model.roi_heads.mask_predictor.conv5_mask.in_channels,256,3)
    return model

def semantic_map(model,image,size):
    torch,device=runtime()
    rgb=cv2.cvtColor(cv2.resize(image,(size,size)),cv2.COLOR_BGR2RGB)
    x=torch.from_numpy(rgb).permute(2,0,1)[None].float().div(255).to(device)
    with torch.inference_mode(): p=torch.softmax(model(x),1)[0].cpu().numpy()
    return np.stack([cv2.resize(v,(image.shape[1],image.shape[0])) for v in p])

def semantic_instances(prob,fg=.5,spacing=9,min_area=25,method='watershed'):
    from scipy import ndimage as ndi
    from skimage.segmentation import watershed
    out=[]; sem=prob.argmax(0)
    for c in (1,2):
        mask=(sem==c)&(prob[c]>=fg)
        if method=='components': labels,_=ndi.label(mask)
        else:
            dist=ndi.distance_transform_edt(mask)
            markers,_=ndi.label((dist==ndi.maximum_filter(dist,spacing))&(dist>2))
            if not markers.max(): markers,_=ndi.label(mask)
            labels=watershed(-dist,markers,mask=mask)
        for k in range(1,int(labels.max())+1):
            m=labels==k
            if m.sum()>=min_area: out.append((c-1,float(prob[c][m].mean()),m))
    return out

print(pd.DataFrame(RECIPES)[['name','init','size','epochs','batch','lr']].to_string(index=False))

       name           init  size  epochs  batch    lr
yolo26n_seg yolo26n-seg.pt   960     100      8 0.001
yolo26s_seg yolo26s-seg.pt   960     100      8 0.001
      mrcnn   COCO_DEFAULT   960      12      2 0.005
    unet_ws         random   512      40      8 0.001
yolo26n_det     yolo26n.pt   960     100      8 0.001
yolo26s_det     yolo26s.pt   960     100      8 0.001


In [7]:
def training_dataset(root,kind,size):
    torch,_=runtime()
    class Dataset(torch.utils.data.Dataset):
        def __init__(self): self.images=sorted((Path(root)/'train/images').glob('*'))
        def __len__(self): return len(self.images)
        def __getitem__(self,i):
            ip=self.images[i]; image=cv2.imread(str(ip)); h,w=image.shape[:2]
            masks,boxes,labels=[],[],[]; sem=np.zeros((h,w),np.uint8)
            for line in (Path(root)/'train/labels'/(ip.stem+'.txt')).read_text().splitlines():
                p=line.split(); xy=np.asarray(p[1:],float).reshape(-1,2)*[w,h]
                m=np.zeros((h,w),np.uint8); cv2.fillPoly(m,[xy.astype(np.int32)],1)
                if not m.any(): raise ValueError('Empty training mask')
                ys,xs=np.where(m); boxes.append([xs.min(),ys.min(),xs.max()+1,ys.max()+1]); masks.append(m); labels.append(int(p[0])+1)
                if np.logical_and(sem>0,m>0).any():
                    sem[np.logical_and(sem>0,m>0)]=255
                sem[(m>0)&(sem!=255)]=int(p[0])+1
            if kind=='unet':
                rgb=cv2.cvtColor(cv2.resize(image,(size,size)),cv2.COLOR_BGR2RGB)
                target=torch.from_numpy(cv2.resize(sem,(size,size),interpolation=cv2.INTER_NEAREST).astype(np.int64))
            else:
                rgb=cv2.cvtColor(image,cv2.COLOR_BGR2RGB)
                target=dict(boxes=torch.as_tensor(boxes,dtype=torch.float32).reshape(-1,4),labels=torch.as_tensor(labels,dtype=torch.int64),masks=torch.as_tensor(np.stack(masks) if masks else np.zeros((0,h,w),np.uint8)),image_id=torch.tensor([i]))
            return torch.from_numpy(rgb).permute(2,0,1).float()/255.,target
    return Dataset()

def predict_model(model,recipe,gt,post=None):
    result={}; post=post or {}
    torch,device=runtime()
    for st,(ip,gm,gc) in gt.items():
        im=cv2.imread(str(ip)); h,w=im.shape[:2]
        if recipe['kind']=='yolo':
            r=model.predict(im,imgsz=recipe['size'],conf=CONF_FLOOR,max_det=MAX_DETS,retina_masks=True,device=device,verbose=False)[0]
            if recipe['task']=='detect':
                result[st]=[(int(c),float(s),np.asarray(box,float)) for c,s,box in zip(r.boxes.cls.cpu().numpy(),r.boxes.conf.cpu().numpy(),r.boxes.xyxy.cpu().numpy())]
            else:
                result[st]=[] if r.masks is None else [(int(c),float(s),cv2.resize((m>.5).astype(np.uint8),(w,h),interpolation=cv2.INTER_NEAREST).astype(bool)) for c,s,m in zip(r.boxes.cls.cpu().numpy(),r.boxes.conf.cpu().numpy(),r.masks.data.cpu().numpy())]
        elif recipe['kind']=='mrcnn':
            x=torch.from_numpy(cv2.cvtColor(im,cv2.COLOR_BGR2RGB)).permute(2,0,1).float().div(255).to(device)
            with torch.inference_mode(): r=model([x])[0]
            result[st]=[(int(c)-1,float(s),m[0]>.5) for c,s,m in zip(r['labels'].cpu().numpy(),r['scores'].cpu().numpy(),r['masks'].cpu().numpy()) if c in (1,2) and s>=CONF_FLOOR and (m[0]>.5).any()]
        else: result[st]=semantic_instances(semantic_map(model,im,recipe['size']),**post)
    return result

def ensure_coco():
    import importlib.util
    if importlib.util.find_spec('pycocotools') is None:
        raise RuntimeError('Canonical model selection requires pycocotools in this kernel.')

def run_spec(recipe,seed,root=DATA,data_id=DATA_ID):
    init_path=EXP/'weights'/recipe['init'] if recipe['kind']=='yolo' else None
    init_sha=file_sha(init_path) if init_path is not None and init_path.exists() else ('PENDING' if init_path else recipe['init'])
    spec=dict(recipe=recipe,seed=seed,data_id=data_id,root=str(root),evaluator=EVAL_VERSION,source=CODE_ID,versions=versions,initializer_sha=init_sha)
    token=hashlib.sha256(json.dumps(spec,sort_keys=True).encode()).hexdigest()[:20]
    return spec,MODEL_ROOT/(recipe['name']+'_'+token)

CHECKPOINTS={}
VAL_PREDS={}
TEST_PREDS={}
POST_SETTINGS={}
THRESHOLDS={}

In [8]:
def train_yolo(recipe,seed,run_dir,root=DATA,valid=GT_VALID):
    from ultralytics import YOLO
    import shutil
    ensure_coco(); seed_everything(seed)
    init=EXP/'weights'/recipe['init']
    if not init.is_file(): raise FileNotFoundError(f'Pretrained initializer required: {init}')
    intervention=recipe.get('intervention','none')
    if intervention not in ('none','clahe','hflip','affine'): raise ValueError('Unknown training intervention')
    if intervention=='clahe':
        derived=run_dir/'train_clahe'; derived.mkdir()
        for split in ('train','valid','test'):
            for kind in ('images','labels'): (derived/split/kind).mkdir(parents=True)
            for ip in sorted((Path(root)/split/'images').iterdir()):
                if ip.suffix.lower() not in {'.jpg','.jpeg','.png','.bmp'}: continue
                target=derived/split/'images'/ip.name
                if split=='train':
                    image=cv2.imread(str(ip)); lab=cv2.cvtColor(image,cv2.COLOR_BGR2LAB)
                    lab[:,:,0]=cv2.createCLAHE(clipLimit=2.,tileGridSize=(8,8)).apply(lab[:,:,0])
                    if not cv2.imwrite(str(target),cv2.cvtColor(lab,cv2.COLOR_LAB2BGR)): raise OSError('CLAHE image write failed')
                else: shutil.copy2(ip,target)
                lt=Path(root)/split/'labels'/(ip.stem+'.txt'); dest=derived/split/'labels'/lt.name
                shutil.copy2(lt,dest); assert file_sha(lt)==file_sha(dest)
        shutil.copy2(Path(root)/'split_audit.csv',derived/'split_audit.csv')
        root=derived
    cfg=run_dir/'data.yaml'
    cfg.write_text(f'path: {Path(root).resolve()}\ntrain: train/images\nval: valid/images\ntest: test/images\nnames: {CLS}\n')
    model=YOLO(str(init))
    model.train(data=str(cfg),imgsz=recipe['size'],batch=recipe['batch'],epochs=recipe['epochs'],patience=recipe['epochs']+1,seed=seed,optimizer='AdamW',lr0=recipe['lr'],amp=False,deterministic=True,workers=0,save_period=1,mosaic=0,fliplr=.5 if intervention=='hflip' else 0,flipud=0,mixup=0,copy_paste=0,hsv_h=0,hsv_s=0,hsv_v=0,degrees=12 if intervention=='affine' else 0,translate=.08 if intervention=='affine' else 0,scale=.2 if intervention=='affine' else 0,shear=0,perspective=0,project=str(run_dir),name='native',exist_ok=False)
    candidates=sorted((run_dir/'native/weights').glob('epoch*.pt'),key=lambda p:int(p.stem.replace('epoch','')))
    if not candidates: raise RuntimeError('No epoch checkpoints saved; verify save_period support.')
    log=[]; best=-1.; selected=None
    for cp in candidates:
        predictor=YOLO(str(cp)); pr=predict_model(predictor,recipe,valid)
        score=canonical_coco(pr,valid,'bbox' if recipe['task']=='detect' else 'segm')['AP']
        if score is None or not np.isfinite(score): raise ValueError('Canonical validation AP unavailable')
        log.append(dict(epoch=int(cp.stem.replace('epoch',''))+1,val_AP=score))
        if score>best: best=score; selected=cp
        del predictor
    shutil.copy2(selected,run_dir/'selected.pt')
    pd.DataFrame(log).to_csv(run_dir/'selection.csv',index=False)
    return run_dir/'selected.pt'

def load_model(recipe,path):
    torch,device=runtime()
    if recipe['kind']=='yolo':
        from ultralytics import YOLO
        return YOLO(str(path))
    model=make_mrcnn(False) if recipe['kind']=='mrcnn' else make_unet()
    model.load_state_dict(torch.load(path,map_location='cpu',weights_only=True))
    return model.to(device).eval()

In [9]:
def train_torch(recipe,seed,run_dir,root=DATA,valid=GT_VALID):
    ensure_coco(); torch,device=seed_everything(seed)
    F=torch.nn.functional
    model=(make_mrcnn(True) if recipe['kind']=='mrcnn' else make_unet()).to(device)
    ds=training_dataset(root,recipe['kind'],recipe['size'])
    collate=(lambda batch:tuple(zip(*batch))) if recipe['kind']=='mrcnn' else None
    dl=torch.utils.data.DataLoader(ds,batch_size=recipe['batch'],shuffle=True,num_workers=0,collate_fn=collate,drop_last=False)
    optimizer=torch.optim.SGD(model.parameters(),lr=recipe['lr'],momentum=.9,weight_decay=5e-4) if recipe['kind']=='mrcnn' else torch.optim.Adam(model.parameters(),lr=recipe['lr'])
    scheduler=torch.optim.lr_scheduler.MultiStepLR(optimizer,[8,11],gamma=.1) if recipe['kind']=='mrcnn' else None
    best=-1.; log=[]
    for epoch in range(recipe['epochs']):
        model.train(); losses=[]
        for images,target in dl:
            if recipe['kind']=='mrcnn':
                images=[x.to(device) for x in images]; target=[{k:v.to(device) for k,v in t.items()} for t in target]
                loss=sum(model(images,target).values())
            else:
                images,target=images.to(device),target.to(device)
                logits=model(images); valid_px=target!=255
                safe=target.masked_fill(~valid_px,0)
                onehot=F.one_hot(safe,3).permute(0,3,1,2).float()*valid_px[:,None]
                prob=F.softmax(logits,1)*valid_px[:,None]
                inter=(prob*onehot).sum((2,3)); denom=(prob+onehot).sum((2,3))
                dice=1-((2*inter+1e-6)/(denom+1e-6)).mean()
                ce=-(F.log_softmax(logits,1)*onehot).sum()/valid_px.sum().clamp_min(1)
                loss=ce+dice
            if not torch.isfinite(loss): raise RuntimeError('Non-finite training loss; run stopped without claiming success.')
            optimizer.zero_grad(); loss.backward(); optimizer.step(); losses.append(float(loss.detach()))
        if scheduler: scheduler.step()
        model.eval(); pr=predict_model(model,recipe,valid)
        score=canonical_coco(pr,valid)['AP']
        log.append(dict(epoch=epoch+1,loss=float(np.mean(losses)),val_AP=score))
        if score>best:
            best=score; torch.save(model.state_dict(),run_dir/'selected.pt')
        pd.DataFrame(log).to_csv(run_dir/'selection.csv',index=False)
        print(recipe['name'],seed,epoch+1,log[-1])
    return run_dir/'selected.pt'

def train_or_load(recipe,seed,root=DATA,data_id=DATA_ID,valid=GT_VALID):
    spec,run_dir=run_spec(recipe,seed,root,data_id)
    manifest=run_dir/'manifest.json'; cp=run_dir/'selected.pt'
    if manifest.exists():
        stored=json.loads(manifest.read_text())
        assert stored['spec']==spec and stored['checkpoint_sha']==file_sha(cp), 'Stale/incompatible checkpoint'
        return cp
    if not DO_TRAIN: return None
    if HANDOFF is None or HANDOFF['geometry_status']!='shapely validity checked': raise RuntimeError('Training requires matching nb00 geometry-verified handoff; run the final audit in a complete EDA kernel.')
    if run_dir.exists(): raise FileExistsError('Incomplete run exists. Inspect it and choose a new recipe/run identity; never overwrite silently.')
    run_dir.mkdir(parents=True)
    start=time.perf_counter()
    cp=train_yolo(recipe,seed,run_dir,root,valid) if recipe['kind']=='yolo' else train_torch(recipe,seed,run_dir,root,valid)
    write_json(manifest,dict(spec=spec,checkpoint_sha=file_sha(cp),training_seconds=time.perf_counter()-start,train_stems=MANIFEST.loc[MANIFEST.split=='train','stem'].tolist() if root==DATA else None))
    return cp

for recipe in RECIPES:
    for seed in SEEDS:
        cp=train_or_load(recipe,seed)
        if cp: CHECKPOINTS[(recipe['name'],seed)]=cp
        else: print('PENDING training:',recipe['name'],seed)
print('Compatible checkpoints:',len(CHECKPOINTS))

PENDING training: yolo26n_seg 42
PENDING training: yolo26n_seg 123
PENDING training: yolo26n_seg 2026


PENDING training: yolo26s_seg 42


PENDING training: yolo26s_seg 123
PENDING training: yolo26s_seg 2026
PENDING training: mrcnn 42
PENDING training: mrcnn 123
PENDING training: mrcnn 2026
PENDING training: unet_ws 42
PENDING training: unet_ws 123
PENDING training: unet_ws 2026
PENDING training: yolo26n_det 42
PENDING training: yolo26n_det 123
PENDING training: yolo26n_det 2026
PENDING training: yolo26s_det 42
PENDING training: yolo26s_det 123


PENDING training: yolo26s_det 2026
Compatible checkpoints: 0


In [10]:
def cache_predictions(model,recipe,seed,cp,gt,split,post=None):
    if split=='test': log_test_access(recipe['name']+':'+str(seed))
    metadata=dict(dataset_id=DATA_ID,checkpoint_sha=file_sha(cp),recipe=recipe,seed=seed,split=split,stems=sorted(gt),floor=CONF_FLOOR,max_dets=MAX_DETS,post=post or {},source=CODE_ID,format='numeric-npz-v1')
    token=hashlib.sha256(json.dumps(metadata,sort_keys=True).encode()).hexdigest()
    path=CACHE_ROOT/(token+'.npz'); meta=CACHE_ROOT/(token+'.json')
    if path.exists():
        stored=json.loads(meta.read_text()) if meta.exists() else {}
        if stored.get('spec')!=metadata or stored.get('sha256')!=file_sha(path): raise ValueError('Prediction cache provenance/checksum mismatch')
        pr={}
        with np.load(path,allow_pickle=False) as arrays:
            for i,st in enumerate(metadata['stems']):
                pr[st]=[(int(c),float(s),m) for c,s,m in zip(arrays[f'c{i}'],arrays[f's{i}'],arrays[f'm{i}'])]
        validate_predictions(pr,gt,recipe['task'])
        return pr
    if not DO_INFER: return None
    if model is None: raise ValueError('Inference enabled but no model supplied')
    pr=predict_model(model,recipe,gt,post); validate_predictions(pr,gt,recipe['task'])
    payload={}
    for i,st in enumerate(metadata['stems']):
        pl=pr[st]
        payload[f'c{i}']=np.array([p[0] for p in pl],dtype=np.int16)
        payload[f's{i}']=np.array([p[1] for p in pl],dtype=float)
        shape=gt[st][1][0].shape if gt[st][1] else cv2.imread(str(gt[st][0])).shape[:2]
        payload[f'm{i}']=np.stack([p[2] for p in pl]) if pl else np.empty((0,4) if recipe['task']=='detect' else (0,*shape),dtype=float if recipe['task']=='detect' else bool)
    np.savez_compressed(path,**payload); write_json(meta,dict(spec=metadata,sha256=file_sha(path)))
    return pr

for recipe in RECIPES:
    for seed in SEEDS:
        key=(recipe['name'],seed); cp=CHECKPOINTS.get(key)
        if cp is None: continue
        model=load_model(recipe,cp) if DO_INFER else None
        post={}; post_file=cp.parent/'postprocessing.json'
        post_spec=dict(dataset_id=DATA_ID,checkpoint_sha=file_sha(cp),source=CODE_ID,selected_on='valid')
        if recipe['kind']=='unet':
            if post_file.exists():
                stored=json.loads(post_file.read_text())
                if stored['spec']!=post_spec: raise ValueError('Stale postprocessing selection')
                post=stored['settings']
            elif DO_INFER:
                ensure_coco(); best=-1.; trials=[]
                for fg,spacing,min_area,method in itertools.product([.3,.5,.7],[5,9,15],[4,25],['components','watershed']):
                    candidate=dict(fg=fg,spacing=spacing,min_area=min_area,method=method)
                    pr=predict_model(model,recipe,GT_VALID,candidate); score=canonical_coco(pr,GT_VALID)['AP']
                    trials.append(dict(**candidate,val_AP=score))
                    if score>best: best=score; post=candidate
                pd.DataFrame(trials).to_csv(OUT/f'postprocessing_{seed}.csv',index=False)
                write_json(post_file,dict(spec=post_spec,settings=post))
            else:
                print('PENDING validation-only postprocessing:',key); continue
        POST_SETTINGS[key]=post
        pr=cache_predictions(model,recipe,seed,cp,GT_VALID,'valid',post)
        if pr is not None:
            VAL_PREDS[key]=pr; THRESHOLDS[key]=select_thresholds(pr,GT_VALID)
            write_json(OUT/f'thresholds_{recipe["name"]}_{seed}.json',dict(selected_on='valid',thresholds=THRESHOLDS[key],post=post))
            if DO_EVAL_TEST:
                test=cache_predictions(model,recipe,seed,cp,GT_TEST,'test',post)
                if test is not None: TEST_PREDS[key]=test
        del model
        if DO_INFER:
            torch,device=runtime()
            if torch.cuda.is_available(): torch.cuda.empty_cache()
print('Validation/test prediction caches:',len(VAL_PREDS),len(TEST_PREDS))

Validation/test prediction caches: 0 0


## 4. ตารางหลัก — canonical mask/box AP และ operating points จาก validation

Segmentation arms ได้ทั้ง mask/box metrics ส่วน detection arms ได้ box metrics เท่านั้น ไม่เลือก F1 ที่ดีที่สุดจาก test Freeze thresholds รายคลาสที่เลือกบน validation ก่อนประเมิน test คลาสไม่มี GT เป็น N/A การเทียบแบบ paired ต้องใช้ cohort และ representation เดียวกัน Standard deviation ระหว่าง training seeds ไม่ใช่ความแปรปรวนระหว่าง folds

In [11]:
RESULTS=[]; OPERATING=[]
for recipe in RECIPES:
    for seed in SEEDS:
        key=(recipe['name'],seed); pr=TEST_PREDS.get(key)
        if pr is None: continue
        assert DO_EVAL_TEST and key in THRESHOLDS
        box=canonical_coco(pr,GT_TEST,'bbox')
        mask=canonical_coco(pr,GT_TEST,'segm') if recipe['task']=='segment' else {}
        row=dict(model=recipe['name'],seed=seed,protocol='development-inspected grouped gold',dataset_id=DATA_ID,box_AP=box.get('AP'),box_AP50=box.get('AP50'),mask_AP=mask.get('AP'),mask_AP50=mask.get('AP50'),mask_AP75=mask.get('AP75'),AP_small=mask.get('AP_small'),AP_medium=mask.get('AP_medium'),AP_large=mask.get('AP_large'),canonical_status=box['status'])
        RESULTS.append(row)
        ops=operating_metrics(pr,GT_TEST,THRESHOLDS[key]); ops['model']=recipe['name']; ops['seed']=seed
        OPERATING.append(ops)
        write_json(OUT/f'metrics_{recipe["name"]}_{seed}.json',dict(box=box,mask=mask,reference=eval_reference(pr,GT_TEST),threshold_source='valid'))
TABLE=pd.DataFrame(RESULTS)
if len(TABLE):
    TABLE.to_csv(OUT/'main_results.csv',index=False)
    print(TABLE.to_string(index=False))
    stats=TABLE.groupby('model')[['box_AP','mask_AP']].agg(['mean','std','count'])
    stats.to_csv(OUT/'seed_summary.csv'); print(stats)
    pd.concat(OPERATING).to_csv(OUT/'operating_points.csv',index=False)
else: print('PENDING: no compatible grouped-holdout predictions. Old seen-clip scores are not copied into this table.')

PENDING: no compatible grouped-holdout predictions. Old seen-clip scores are not copied into this table.


## 5. Error taxonomy, boundary/semantic quality และ quality slices

Classification/localization/both/duplicate/background/missed เป็นคำอธิบาย errors ที่พบ ไม่ใช่ causal TIDE dAP รายงานทั้งจำนวนและ denominator Boundary IoU คำนวณเฉพาะ instances ที่จับคู่ได้ ต้องอ่านคู่กับ recall Semantic IoU/Dice รวมบริเวณ predicted/GT ที่จับคู่ไม่ได้ โดย ignore GT pixels ที่ต่างคลาสซ้อนกัน ไม่ตัดสินคลาสตามใจ Slice AP ใช้ GT/predictions ทุกตัวบนภาพใน slice ไม่กรองขนาด GT โดยเงียบ ๆ

In [12]:
def diagnose(preds,gt,thresholds,segmentation=True):
    rows=[]; boundaries=[]; semantic=[]
    for st,(ip,gm,gc) in gt.items():
        pl=[p for p in preds.get(st,[]) if p[1]>=thresholds[p[0]]]
        rr,used=match_image(pl,gm,gc); matrix=iou_mat([m for _,_,m in pl],gm)
        for r in rr:
            i=r['pred']; c,s,m=pl[i]
            if r['tp']:
                label='TP'
                if segmentation: boundaries.append(dict(stem=st,cls=CLS[c],IoU=r['iou'],BoundaryIoU=boundary_iou(m,gm[r['gt']])))
            else:
                same=[j for j,x in enumerate(gc) if x==c]; other=[j for j,x in enumerate(gc) if x!=c]
                si=max(matrix[i,same],default=0); oi=max(matrix[i,other],default=0)
                label='duplicate' if si>=.5 else ('classification' if oi>=.5 else ('localization' if si>=.1 else ('both' if oi>=.1 else 'background')))
            rows.append(dict(stem=st,group=GROUP_MAP[st],cls=CLS[c],error=label,score=s))
        for j,c in enumerate(gc):
            if j not in used: rows.append(dict(stem=st,group=GROUP_MAP[st],cls=CLS[c],error='missed',score=None))
        if segmentation:
            shape=gm[0].shape if gm else cv2.imread(str(ip)).shape[:2]
            gold=np.zeros(shape,np.uint8); pred=np.zeros(shape,np.uint8)
            masks=[np.logical_or.reduce([m for m,c in zip(gm,gc) if c==k]) if k in gc else np.zeros(shape,bool) for k in (0,1)]
            ambiguous=masks[0]&masks[1]
            for k in (0,1): gold[masks[k]]=k+1
            gold[ambiguous]=255
            for c,s,m in sorted(pl,key=lambda p:p[1]): pred[np.asarray(m,bool)]=c+1
            for k in (1,2):
                g=(gold==k)&~ambiguous; p=(pred==k)&~ambiguous
                semantic.append(dict(stem=st,cls=CLS[k-1],intersection=int((g&p).sum()),union=int((g|p).sum()),denominator=int(g.sum()+p.sum())))
    return pd.DataFrame(rows,columns=['stem','group','cls','error','score']),pd.DataFrame(boundaries,columns=['stem','cls','IoU','BoundaryIoU']),pd.DataFrame(semantic,columns=['stem','cls','intersection','union','denominator'])

ERRORS={}; GROUP_RESULTS=[]; SLICE_RESULTS=[]
quality=pd.read_csv(QUALITY_PATH) if QUALITY_PATH else pd.DataFrame()
if len(quality):
    assert quality.stem.is_unique and set(quality.stem)==set(MANIFEST.stem), 'Quality cohort differs from dataset'
    assert quality.set_index('stem').group.to_dict()==GROUP_MAP
for key,pr in TEST_PREDS.items():
    recipe=next(r for r in RECIPES if r['name']==key[0]); segmentation=recipe['task']=='segment'
    er,bi,sem=diagnose(pr,GT_TEST,THRESHOLDS[key],segmentation); ERRORS[key]=er
    tag=f'{key[0]}_{key[1]}'
    er.to_csv(OUT/f'errors_{tag}.csv',index=False); bi.to_csv(OUT/f'boundary_{tag}.csv',index=False)
    if len(sem):
        totals=sem.groupby('cls')[['intersection','union','denominator']].sum()
        totals['IoU']=totals.intersection/totals.union.replace(0,np.nan)
        totals['Dice']=2*totals.intersection/totals.denominator.replace(0,np.nan)
        totals.to_csv(OUT/f'semantic_{tag}.csv')
    for group in sorted({GROUP_MAP[st] for st in GT_TEST}):
        cohort={st:g for st,g in GT_TEST.items() if GROUP_MAP[st]==group}; subset={st:pr[st] for st in cohort}
        ops=operating_metrics(subset,cohort,THRESHOLDS[key]); ops['group']=group; ops['model']=key[0]; ops['seed']=key[1]; ops['images']=len(cohort)
        GROUP_RESULTS.append(ops)
    if len(quality):
        for flag in [c for c in quality if c.startswith('flag_')]:
            assert quality[flag].isin([True,False]).all(), f'Invalid quality flag encoding: {flag}'
            for value in (False,True):
                stems=sorted(set(quality.loc[quality[flag]==value,'stem'])&set(GT_TEST))
                if not stems: continue
                cohort={st:GT_TEST[st] for st in stems}; subset={st:pr[st] for st in stems}
                ref=eval_reference(subset,cohort)
                SLICE_RESULTS.append(dict(model=key[0],seed=key[1],slice=flag,value=value,images=len(stems),groups=len({GROUP_MAP[st] for st in stems}),reference_AP=ref['AP_macro'],representation='mask' if segmentation else 'box'))
if GROUP_RESULTS: pd.concat(GROUP_RESULTS).to_csv(OUT/'per_group_operating.csv',index=False)
if SLICE_RESULTS: pd.DataFrame(SLICE_RESULTS).to_csv(OUT/'quality_slices.csv',index=False)
print('Error/slice analysis completed for',len(ERRORS),'compatible runs. No stale disk flags are used.')

Error/slice analysis completed for 0 compatible runs. No stale disk flags are used.


## 6. รูปสำหรับรายงาน — เลือกซ้ำได้ ภาพซ้อนราย instance และ N/A

PR/AP-vs-IoU ใช้ inline diagnostic evaluator ส่วนคะแนนหลักอยู่ canonical table Gallery เลือกไม่เกินหนึ่งภาพต่อ merged group แล้วเรียงตาม errors ที่พบมากที่สุด ต้องเปิดเผยวิธีเลือกและไม่ใช้เพื่อ tuning แสดง instance IDs/confidence ของ GT/prediction ช่อง heatmap ที่ไม่มี GT เป็น N/A ไม่ใช่ศูนย์สีแดง Training curves แสดง canonical validation AP แยกจาก native loss

In [13]:
if TEST_PREDS:
    fig,axes=plt.subplots(1,2,figsize=(11,4))
    for key,pr in TEST_PREDS.items():
        for c,ax in enumerate(axes):
            curve=pr_curve(pr,GT_TEST,c)
            ax.step(curve['recall'],curve['precision'],where='post',label=f'{key[0]} seed {key[1]}')
            ax.set(xlabel='Recall',ylabel='Precision',title=CLS[c],xlim=(0,1),ylim=(0,1)); ax.legend(fontsize=6)
    savefig('PR_by_class')
    fig,ax=plt.subplots(figsize=(8,4))
    for key,pr in TEST_PREDS.items():
        values=[np.mean([v for c in (0,1) if (v:=pr_curve(pr,GT_TEST,c,t)['ap']) is not None]) for t in np.linspace(.5,.95,10)]
        ax.plot(np.linspace(.5,.95,10),values,'o-',label=f'{key[0]} {key[1]}')
    ax.set(xlabel='IoU threshold',ylabel='Reference 101-point AP',ylim=(0,1)); ax.legend(fontsize=7); savefig('AP_vs_IoU')
    for key,errors in ERRORS.items():
        if len(errors):
            counts=errors.groupby(['error','cls']).size().unstack(fill_value=0)
            counts.plot.bar(figsize=(9,4)); plt.ylabel('Count'); plt.title(f'Descriptive error taxonomy: {key}'); savefig(f'errors_{key[0]}_{key[1]}')
    if GROUP_RESULTS:
        group_table=pd.concat(GROUP_RESULTS)
        for model in group_table.model.unique():
            sub=group_table[group_table.model==model]
            pivot=sub.pivot_table(index='group',columns='cls',values='recall',dropna=False).reindex(columns=CLS)
            fig,ax=plt.subplots(figsize=(5,max(3,len(pivot)*.3)))
            cmap=plt.get_cmap('viridis').copy(); cmap.set_bad('#dddddd')
            image=ax.imshow(np.ma.masked_invalid(pivot.to_numpy(float)),vmin=0,vmax=1,cmap=cmap,aspect='auto')
            ax.set_xticks(range(2),CLS); ax.set_yticks(range(len(pivot)),pivot.index,fontsize=6)
            for i in range(len(pivot)):
                for j in range(2):
                    n=int(sub[(sub.group==pivot.index[i])&(sub.cls==CLS[j])][['tp','fn']].sum().sum())
                    val=pivot.iloc[i,j]; ax.text(j,i,'N/A' if pd.isna(val) else f'{val:.2f}\nn={n}',ha='center',va='center',fontsize=6)
            fig.colorbar(image,ax=ax,label='Recall (validation threshold)'); savefig('recall_'+model)

    reference=next(iter(TEST_PREDS)); er=ERRORS[reference]
    ranked=er[er.error!='TP'].groupby('stem').size().sort_values(ascending=False) if len(er) else pd.Series(dtype=float)
    picks=[]; used_groups=set()
    for st in list(ranked.index)+sorted(GT_TEST):
        if GROUP_MAP[st] not in used_groups:
            picks.append(st); used_groups.add(GROUP_MAP[st])
        if len(picks)==6: break
    segkeys=[k for k in TEST_PREDS if all(np.asarray(m).ndim==2 for pl in TEST_PREDS[k].values() for _,_,m in pl)][:4]
    if segkeys and picks:
        fig,axes=plt.subplots(len(picks),len(segkeys)+1,figsize=(4*(len(segkeys)+1),3*len(picks)),squeeze=False)
        for row,st in enumerate(picks):
            ip,gm,gc=GT_TEST[st]; im=cv2.cvtColor(cv2.imread(str(ip)),cv2.COLOR_BGR2RGB)
            for col,key in enumerate([None]+segkeys):
                ax=axes[row,col]; ax.imshow(im)
                pl=[(c,1.,m) for c,m in zip(gc,gm)] if key is None else [p for p in TEST_PREDS[key].get(st,[]) if p[1]>=THRESHOLDS[key][p[0]]]
                for i,(c,s,m) in enumerate(pl):
                    color=['#0072B2','#D55E00'][c]
                    ax.contour(m,levels=[.5],colors=[color],linewidths=1)
                    ys,xs=np.where(m)
                    if len(xs): ax.text(xs.mean(),ys.mean(),f'{i}:{CLS[c]} {s:.2f}',fontsize=6,color=color,bbox=dict(facecolor='white',alpha=.7,edgecolor='none'))
                ax.set_title(('GT' if key is None else str(key))+' | '+st[:12],fontsize=7); ax.axis('off')
        plt.suptitle('Error-ranked gallery; one image per merged group (not a random performance sample)')
        savefig('instance_gallery')
else: print('Figures pending verified prediction caches.')

fig,ax=plt.subplots(figsize=(9,4)); found=False
for key,cp in CHECKPOINTS.items():
    selection=pd.read_csv(cp.parent/'selection.csv')
    ax.plot(selection.epoch,selection.val_AP,label=str(key)); found=True
if found:
    ax.set(xlabel='Epoch',ylabel='Canonical full-validation AP'); ax.legend(fontsize=7); savefig('validation_training_curves')
else: plt.close(fig)

Figures pending verified prediction caches.


## 7. Paired cluster bootstrap — uncertainty ไม่ใช่นับเฟรมเป็นตัวอย่างอิสระ

สุ่ม merged groups แบบใส่คืน ใช้ draw เดียวกันทุกโมเดล คำนวณ AP ใหม่บนภาพที่ทำสำเนาพร้อม ID ใหม่ รายงาน paired delta intervals และ class coverage เป็น uncertainty แบบมีเงื่อนไขบน grouped test ที่มี ไม่ได้สร้างไซต์/เหตุการณ์ใหม่หรือแทนการฝึกหลาย seeds เปิดเผย group น้อยและ resamples ที่ขาดคลาส เปรียบเทียบ segmentation กับ segmentation หรือ detection กับ detection เท่านั้น

In [14]:
def paired_bootstrap(left,right,gt,group_map,n=500,seed=42):
    groups=sorted({group_map[st] for st in gt}); rng=np.random.default_rng(seed)
    if len(groups)<2: return dict(status='BLOCKED: fewer than two merged groups; no cluster CI is meaningful')
    validate_predictions(left,gt); validate_predictions(right,gt)
    draws=[]; missing=0
    for _ in range(n):
        cohort={}; a={}; b={}
        for copy,g in enumerate(rng.choice(groups,len(groups),replace=True)):
            for st,item in gt.items():
                if group_map[st]!=g: continue
                new=f'{copy}:{st}'; cohort[new]=item; a[new]=left.get(st,[]); b[new]=right.get(st,[])
        classes={c for _,gm,gc in cohort.values() for c in gc}
        if classes!={0,1}: missing+=1; continue
        draws.append(eval_reference(a,cohort)['AP_macro']-eval_reference(b,cohort)['AP_macro'])
    if not draws: return dict(status='BLOCKED: no class-complete bootstrap replicates',missing=missing)
    return dict(point_delta=eval_reference(left,gt)['AP_macro']-eval_reference(right,gt)['AP_macro'],delta_mean=float(np.mean(draws)),interval_scope='conditional on class-complete resamples; exploratory, not multiplicity-adjusted',ci_low=float(np.quantile(draws,.025)),ci_high=float(np.quantile(draws,.975)),replicates=len(draws),missing_class_replicates=missing,groups=len(groups),metric='reference AP50:95',seed=seed)

BOOTSTRAP=[]
if DO_BOOTSTRAP:
    for a,b in itertools.combinations(TEST_PREDS,2):
        ra=next(r for r in RECIPES if r['name']==a[0]); rb=next(r for r in RECIPES if r['name']==b[0])
        if a[1]!=b[1] or ra['task']!=rb['task']: continue
        result=paired_bootstrap(TEST_PREDS[a],TEST_PREDS[b],GT_TEST,GROUP_MAP)
        BOOTSTRAP.append(dict(left=str(a),right=str(b),**result))
    write_json(OUT/'paired_bootstrap.json',BOOTSTRAP)
    usable=[x for x in BOOTSTRAP if 'ci_low' in x]
    if usable:
        fig,ax=plt.subplots(figsize=(9,max(3,len(usable)*.4)))
        for i,r in enumerate(usable):
            ax.plot([r['ci_low'],r['ci_high']],[i,i],color='#0072B2'); ax.scatter(r['delta_mean'],i)
        ax.axvline(0,color='black',ls='--'); ax.set_yticks(range(len(usable)),[x['left']+' minus '+x['right'] for x in usable],fontsize=7)
        ax.set_xlabel('Paired reference AP difference, group-bootstrap 95% interval'); savefig('paired_differences')
else: print('Group bootstrap disabled; no confidence intervals claimed.')

Group bootstrap disabled; no confidence intervals claimed.


## 8. วิเคราะห์เพิ่ม — validation calibration, boundary และ class confusion

ใช้ validation วิเคราะห์ confidence และเลือก operating policy Detection reliability ที่นี่คือ `P(class-correct IoU≥0.5 match | score)` ภายใต้ score-ordered one-to-one matching ไม่ใช่ probability ที่ calibrated แล้วของการมีวัสดุ ECE ขึ้นกับ score floor/bins Reliability plot/Brier score ไม่ครอบคลุม GT ที่พลาด จึงต้องรายงาน recall ด้วย

Class confusion จับคู่แบบไม่สนคลาสที่ IoU≥0.5 แถว/คอลัมน์ที่สามหมายถึง predictions ที่ไม่จับคู่/GT ที่พลาด ไม่ใช่ pixel background Boundary distributions เป็น conditional บน mask matches Raw Boundary IoU ไม่ใช่ Boundary AP ไม่ใช้ test diagnostics เพื่อปรับวิธี

In [15]:
def reliability_table(preds,gt,bins=10):
    observations=[]
    for st,(_,gm,gc) in gt.items():
        matches,_=match_image(preds[st],gm,gc)
        observations.extend(dict(cls=CLS[r['cls']],score=r['score'],correct=r['tp']) for r in matches)
    obs=pd.DataFrame(observations,columns=['cls','score','correct'])
    rows=[]
    for cn in CLS:
        sub=obs[obs.cls==cn].copy()
        sub['bin']=np.minimum((sub.score*bins).astype(int),bins-1)
        for b in range(bins):
            v=sub[sub.bin==b]
            rows.append(dict(cls=cn,bin=b,n=len(v),confidence=float(v.score.mean()) if len(v) else None,accuracy=float(v.correct.mean()) if len(v) else None))
    frame=pd.DataFrame(rows); summary=[]
    for cn in CLS:
        v=frame[frame.cls==cn]; raw=obs[obs.cls==cn]
        summary.append(dict(cls=cn,n=len(raw),ECE=float((v.n*(v.confidence-v.accuracy).abs()).sum()/len(raw)) if len(raw) else None,Brier=float(((raw.score-raw.correct)**2).mean()) if len(raw) else None,split='valid',score_floor=CONF_FLOOR))
    return frame,pd.DataFrame(summary)

def instance_confusion(preds,gt,thresholds):
    counts=np.zeros((3,3),int)
    for st,(_,gm,gc) in gt.items():
        pl=[p for p in preds[st] if p[1]>=thresholds[p[0]]]
        rr,used=match_image(pl,gm,gc,class_aware=False)
        for r in rr: counts[gc[r['gt']] if r['gt'] is not None else 2,r['cls']]+=1
        for j,c in enumerate(gc):
            if j not in used: counts[c,2]+=1
    return counts

for key,pr in VAL_PREDS.items():
    tag=f'{key[0]}_{key[1]}'; bins,calibration=reliability_table(pr,GT_VALID)
    bins.to_csv(OUT/f'reliability_bins_{tag}.csv',index=False); calibration.to_csv(OUT/f'calibration_{tag}.csv',index=False)
    fig,ax=plt.subplots(figsize=(5,5)); ax.plot([0,1],[0,1],'--',color='gray',label='Identity')
    for cn,marker in zip(CLS,['o','s']):
        sub=bins[(bins.cls==cn)&(bins.n>0)]
        ax.plot(sub.confidence,sub.accuracy,marker=marker,label=f'{cn}: n={int(sub.n.sum())}')
    ax.set(xlim=(0,1),ylim=(0,1),xlabel='Mean detection score',ylabel='Matched detection fraction',title=f'Validation reliability: {tag}'); ax.legend(); savefig('reliability_'+tag)
for key,pr in TEST_PREDS.items():
    tag=f'{key[0]}_{key[1]}'; counts=instance_confusion(pr,GT_TEST,THRESHOLDS[key])
    pd.DataFrame(counts,index=CLS+['unmatched prediction'],columns=CLS+['missed GT']).to_csv(OUT/f'confusion_{tag}.csv')
    fig,ax=plt.subplots(figsize=(6,5)); image=ax.imshow(counts,cmap='Blues')
    ax.set_xticks(range(3),CLS+['missed GT']); ax.set_yticks(range(3),CLS+['unmatched pred'])
    for i in range(3):
        for j in range(3): ax.text(j,i,'N/A' if (i,j)==(2,2) else str(counts[i,j]),ha='center',va='center')
    ax.set(xlabel='Predicted class / missing',ylabel='GT class / unmatched',title=f'Instance confusion: {tag}'); fig.colorbar(image,ax=ax,label='Count'); savefig('confusion_'+tag)
    recipe=next(r for r in RECIPES if r['name']==key[0])
    if recipe['task']=='segment':
        _,boundary,_=diagnose(pr,GT_TEST,THRESHOLDS[key])
        if len(boundary):
            fig,ax=plt.subplots(figsize=(6,4))
            for cn,marker in zip(CLS,['o','s']):
                sub=boundary[boundary.cls==cn]
                ax.scatter(sub.IoU,sub.BoundaryIoU,s=12,alpha=.4,marker=marker,label=f'{cn}, matched n={len(sub)}')
            ax.set(xlim=(0,1),ylim=(0,1),xlabel='Mask IoU (matched instances only)',ylabel='Boundary IoU',title=tag); ax.legend(); savefig('boundary_'+tag)
if not VAL_PREDS: print('PENDING: validation reliability requires provenance-verified predictions.')

PENDING: validation reliability requires provenance-verified predictions.


## 9. Controlled ablations และหลาย seeds — validation ก่อน

การทดลอง fixed-split ที่ implement แล้ว: YOLO26n segmentation 960 ไม่มี online augmentation เป็น control เปลี่ยนทีละปัจจัยเป็น resolution 640, CLAHE เฉพาะ train, horizontal flip เฉพาะ train และ affine เฉพาะ train Validation/test pixels และ annotations ไม่เปลี่ยน วัด **training-data intervention** ไม่ใช่ preprocessing ทั้ง train/deployment Affine arm ใช้ polygon transform ของ Ultralytics รุ่นที่ติดตั้ง ต้องตรวจ label/clipping ของรุ่นนั้นก่อนสรุป ไม่มี offline resplit ใน runner นี้

ใช้ epochs/optimizer/seeds เดียวกัน ประเมิน full validation ก่อน optional development-test รันทุก arms ที่วางแผน ไม่เลือกเฉพาะผลที่ดูดี Native multi-model recipes เป็น practical benchmark ไม่ใช่ architecture-controlled comparison U-Net components กับ watershed บันทึกผ่าน validation postprocessing sweep เพิ่ม seeds ได้ก่อนเริ่มทดลอง; 3 seeds เป็นจุดเริ่ม ไม่ใช่หลักฐานว่าผลเสถียร

CV พร้อม offline augmentation เป็นงานถัดไปจากส่วน 17 ใน notebook 00 **ยังไม่ได้ต่อกับ runner นี้** การเทียบ control/aug ของ CV ต้อง match optimizer steps/validation selection steps ไม่ใช่ถือว่า epochs เท่ากันเพียงพอ

In [16]:
DO_ABLATIONS=False
base=dict(next(r for r in RECIPES if r['name']=='yolo26n_seg'))
ABLATIONS=[dict(base,name='ablation_control',intervention='none'),dict(base,name='ablation_640',size=640,intervention='none'),dict(base,name='ablation_clahe',intervention='clahe'),dict(base,name='ablation_hflip',intervention='hflip'),dict(base,name='ablation_affine',intervention='affine')]
ABLATION_PREDS={}; ABLATION_RESULTS=[]
write_json(OUT/'ablation_plan.json',dict(arms=ABLATIONS,seeds=SEEDS,selection='full valid canonical AP',test_status='development-inspected only',vary_one_factor=True))
if DO_ABLATIONS:
    if not DO_TRAIN and not DO_INFER: print('Ablations requested in read-only mode; only compatible caches/checkpoints can be used.')
    for recipe in ABLATIONS:
        for seed in SEEDS:
            cp=train_or_load(recipe,seed)
            if cp is None: print('PENDING ablation:',recipe['name'],seed); continue
            model=load_model(recipe,cp) if DO_INFER else None
            valid=cache_predictions(model,recipe,seed,cp,GT_VALID,'valid')
            if valid is not None:
                metrics=canonical_coco(valid,GT_VALID)
                ABLATION_RESULTS.append(dict(arm=recipe['name'],seed=seed,split='valid',AP=metrics.get('AP'),status=metrics['status']))
                if DO_EVAL_TEST:
                    test=cache_predictions(model,recipe,seed,cp,GT_TEST,'test')
                    if test is not None:
                        key=(recipe['name'],seed); ABLATION_PREDS[key]=test
                        metrics=canonical_coco(test,GT_TEST)
                        ABLATION_RESULTS.append(dict(arm=recipe['name'],seed=seed,split='development_test',AP=metrics.get('AP'),status=metrics['status']))
            del model
            if DO_INFER:
                torch,device=runtime()
                if torch.cuda.is_available(): torch.cuda.empty_cache()
    ablation_table=pd.DataFrame(ABLATION_RESULTS)
    if len(ablation_table):
        ablation_table.to_csv(OUT/'ablations.csv',index=False)
        fig,ax=plt.subplots(figsize=(9,4))
        for split,marker in [('valid','o'),('development_test','s')]:
            sub=ablation_table[ablation_table.split==split]
            for i,arm in enumerate([r['name'] for r in ABLATIONS]):
                values=sub.loc[sub.arm==arm,'AP'].dropna()
                if len(values): ax.scatter(np.full(len(values),i),values,marker=marker,label=split if i==0 else None)
        ax.set_xticks(range(len(ABLATIONS)),[r['name'].replace('ablation_','') for r in ABLATIONS],rotation=20)
        ax.set(ylabel='Canonical mask AP',ylim=(0,1),title='Ablation observations: individual seeds, not fold CI'); ax.legend(); savefig('ablations')
else: print('Controlled ablations disabled; no improvement claim is made.')

Controlled ablations disabled; no improvement claim is made.


## 10. วัดประสิทธิภาพบน hardware จริง ไม่ประมาณแทน Pi5

Batch-one latency รวม preprocessing, inference, CPU transfer และ postprocessing ของ notebook แต่ไม่รวมอ่าน/ถอดรหัสภาพ วัดภาพ validation ที่สุ่มด้วย seed หลัง warmup และ synchronize CUDA/ROCm ก่อน/หลังจับเวลา บันทึก median/p95/p99, peak allocated memory, ขนาด initializer/checkpoint และ hardware จริง Percentiles อธิบาย workload นี้ ไม่ใช่ production SLO ที่รับรองแล้ว ต้อง benchmark deployment target แยก ไม่ใช้ตัวคูณแปลง GPU เป็น Pi

ขอบเขต implementation ปัจจุบัน: benchmark ของ YOLO/Mask R-CNN ยังไม่ได้รวมการแปลงผลเป็นรูปแบบ mask/box ทั้งหมดแบบ `predict_model` จึงต้องตรวจให้ตรงกันก่อนอ้าง end-to-end postprocessing parity

In [17]:
DO_BENCHMARK=False
EFFICIENCY=[]
def benchmark_model(recipe,cp,post=None,iterations=50,warmup=10):
    torch,device=seed_everything(42); model=load_model(recipe,cp)
    rng=np.random.default_rng(42); stems=sorted(GT_VALID)
    if not stems: raise ValueError('No validation images available')
    selected=[stems[i] for i in rng.choice(len(stems),min(8,len(stems)),replace=False)]
    images=[cv2.imread(str(GT_VALID[st][0])) for st in selected]
    if any(im is None for im in images): raise ValueError('Image decode failed')
    def sync():
        if torch.cuda.is_available(): torch.cuda.synchronize()
    def step(im):
        if recipe['kind']=='yolo':
            return model.predict(im,imgsz=recipe['size'],conf=CONF_FLOOR,max_det=MAX_DETS,retina_masks=True,device=device,verbose=False)[0].cpu()
        if recipe['kind']=='mrcnn':
            x=torch.from_numpy(cv2.cvtColor(im,cv2.COLOR_BGR2RGB)).permute(2,0,1).float().div(255).to(device)
            with torch.inference_mode(): return {k:v.cpu() for k,v in model([x])[0].items()}
        return semantic_instances(semantic_map(model,im,recipe['size']),**(post or {}))
    for i in range(warmup): step(images[i%len(images)])
    sync()
    if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()
    times=[]
    for i in range(iterations):
        sync(); start=time.perf_counter(); step(images[i%len(images)]); sync(); times.append((time.perf_counter()-start)*1000)
    module=model.model if recipe['kind']=='yolo' else model
    record=dict(model=recipe['name'],checkpoint_sha=file_sha(cp),device=device,hardware=torch.cuda.get_device_name() if torch.cuda.is_available() else platform.processor() or platform.machine(),torch=torch.__version__,rocm=getattr(torch.version,'hip',None),cuda=getattr(torch.version,'cuda',None),precision='FP32',batch=1,input_size=recipe['size'],n=iterations,warmup=warmup,validation_images=selected,median_ms=float(np.median(times)),p95_ms=float(np.quantile(times,.95)),p99_ms=float(np.quantile(times,.99)),params_M=sum(p.numel() for p in module.parameters())/1e6,checkpoint_MB=cp.stat().st_size/1e6,peak_allocated_MB=torch.cuda.max_memory_allocated()/1e6 if torch.cuda.is_available() else None)
    del model
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return record
if DO_BENCHMARK:
    for recipe in RECIPES:
        key=(recipe['name'],SEEDS[0]); cp=CHECKPOINTS.get(key)
        if cp is None: print('PENDING benchmark checkpoint:',key); continue
        if recipe['kind']=='unet' and key not in POST_SETTINGS: print('PENDING validation postprocessing:',key); continue
        EFFICIENCY.append(benchmark_model(recipe,cp,POST_SETTINGS.get(key)))
    write_json(OUT/'efficiency_measured.json',EFFICIENCY)
    if EFFICIENCY: pd.DataFrame(EFFICIENCY).to_csv(OUT/'efficiency_measured.csv',index=False)
else: print('Efficiency benchmark disabled. Pi5 throughput and energy remain unmeasured.')

Efficiency benchmark disabled. Pi5 throughput and energy remain unmeasured.


## 11. Export gate และความพร้อมใช้งานจริง

Export ต้องเปิดเองและใช้ checkpoint ที่ผ่าน provenance ONNX syntax validation ไม่ใช่ numerical/task-metric parity ก่อน production ให้รัน target runtime บน validation cohort เดิม เทียบ boxes/masks/task metrics ตาม tolerance ที่กำหนดล่วงหน้า ทดสอบ input เสีย/ว่าง และวัด target hardware จริง INT8 calibration ใช้ train ไม่ใช้ test Notebook นี้ไม่อ้าง legal certification, NIST compliance, Raspberry Pi throughput, OOD detection หรือความปลอดภัยของระบบอัตโนมัติ

สร้าง model card/evidence register จาก artifacts ปัจจุบัน หลักฐานที่ยังขาดสำหรับ production ได้แก่ independent labels, negative/background scenes, event metadata, target latency, runtime parity และผู้รับผิดชอบ เป็นข้อจำกัดการนำไปใช้ **ไม่ใช่ข้อกำหนดให้สร้าง untouched confirmatory test ก่อน development CV training**

In [18]:
EXPORT_RECORDS=[]
if DO_EXPORT:
    import onnx
    from ultralytics import YOLO
    for recipe in RECIPES:
        key=(recipe['name'],SEEDS[0]); cp=CHECKPOINTS.get(key)
        if recipe['kind']!='yolo' or cp is None: continue
        exported=Path(YOLO(str(cp)).export(format='onnx',imgsz=recipe['size'],batch=1,dynamic=False,simplify=False,half=False))
        onnx.checker.check_model(onnx.load(str(exported)))
        EXPORT_RECORDS.append(dict(model=recipe['name'],checkpoint_sha=file_sha(cp),onnx=str(exported),onnx_sha=file_sha(exported),syntax='checked',runtime_parity='PENDING',task_metric_parity='PENDING',target_hardware='PENDING'))
    write_json(OUT/'exports.json',EXPORT_RECORDS)
else: print('Export disabled; no deployment parity claimed.')

EVIDENCE=pd.DataFrame([
    dict(requirement='Image/label identity and merged-group disjointness',status='PASS',evidence='dataset_manifest.csv',scope='filename-derived groups, not independent events'),
    dict(requirement='Canonical COCO evaluator',status='AVAILABLE' if versions['pycocotools']!='NOT INSTALLED' else 'BLOCKED',evidence='metric self-check cell',scope='noncrowd dataset'),
    dict(requirement='Completed planned arms/seeds',status='PASS' if len(TABLE)==len(RECIPES)*len(SEEDS) else 'PENDING',evidence='main_results.csv',scope='development-inspected grouped test'),
    dict(requirement='Independent human labels and adjudication',status='PENDING',evidence='nb00 reviewer queue',scope='pseudo-GT cannot establish annotation correctness'),
    dict(requirement='New event/site confirmatory test',status='DEFERRED',evidence='owner metadata deferral',scope='no external validity claim'),
    dict(requirement='Negative/background-scene evaluation',status='PENDING',evidence='nb00 final distribution audit',scope='FP rate on target background scenes unknown'),
    dict(requirement='Controlled ablations',status='RUN' if ABLATION_RESULTS else 'PENDING',evidence='ablations.csv',scope='verify full planned arm coverage before concluding'),
    dict(requirement='Conditional cluster uncertainty',status='RUN' if BOOTSTRAP else 'PENDING',evidence='paired_bootstrap.json',scope='not seed variability; sparse groups disclosed'),
    dict(requirement='Actual execution-hardware efficiency',status='MEASURED' if EFFICIENCY else 'PENDING',evidence='efficiency_measured.json',scope='not Pi5 or field throughput'),
    dict(requirement='Deployment runtime/task parity',status='PENDING',evidence='exports.json',scope='ONNX syntax alone insufficient'),
    dict(requirement='Monitoring owner, rollback, acceptance limits',status='PENDING',evidence='model card',scope='human operational sign-off required')
])
EVIDENCE.to_csv(OUT/'readiness_evidence.csv',index=False); display(EVIDENCE)
MODEL_CARD=dict(task='Gravel/sand annotated material-region segmentation and detection',intended_use='Research decision support; supervised visual inspection',excluded_use=['individual-grain sizing','mass or volume estimation','autonomous consequential field decisions'],dataset_id=DATA_ID,source_id=CODE_ID,test_status='development-inspected grouped benchmark',group_semantics='merged filename/dHash groups; events and sites unverified',labels='pseudo-GT; independent human adjudication pending',recipes=RECIPES,seeds=SEEDS,threshold_source='full valid',results=RESULTS,measured_efficiency=EFFICIENCY,limitations=EVIDENCE[EVIDENCE.status!='PASS'].to_dict('records'),monitoring={'observe':['brightness/blur against fixed training reference','class score and empty-prediction rates','annotated sample precision/recall and boundary quality','latency/error rate by hardware/runtime'],'owner':None,'cadence':None,'alert_limits':None,'rollback':'retain known checkpoint and escalate to human; never auto-retrain on drift alone'})
write_json(OUT/'model_card.json',MODEL_CARD)

Export disabled; no deployment parity claimed.


,requirement,status,evidence,scope
0,Image/label identity and merged-group disjoint...,PASS,dataset_manifest.csv,"filename-derived groups, not independent events"
1,Canonical COCO evaluator,AVAILABLE,metric self-check cell,noncrowd dataset
2,Completed planned arms/seeds,PENDING,main_results.csv,development-inspected grouped test
3,Independent human labels and adjudication,PENDING,nb00 reviewer queue,pseudo-GT cannot establish annotation correctness
4,New event/site confirmatory test,DEFERRED,owner metadata deferral,no external validity claim
5,Negative/background-scene evaluation,PENDING,nb00 final distribution audit,FP rate on target background scenes unknown
6,Controlled ablations,PENDING,ablations.csv,verify full planned arm coverage before conclu...
7,Conditional cluster uncertainty,PENDING,paired_bootstrap.json,not seed variability; sparse groups disclosed
8,Actual execution-hardware efficiency,PENDING,efficiency_measured.json,not Pi5 or field throughput
9,Deployment runtime/task parity,PENDING,exports.json,ONNX syntax alone insufficient


## 12. Accuracy–efficiency และข้อตกลงการรายงาน

เชื่อม latency ที่วัดจริงกับ model/checkpoint/seed และ dataset ที่ตรงกันเท่านั้น แยก box accuracy กับ mask accuracy เปิดเผยความต่าง resolution/pretraining/optimizer Group-bootstrap CI, training-seed SD และ latency percentiles เป็น uncertainty คนละชนิด ห้ามใช้แทนกัน

ลำดับวิทยานิพนธ์: dataset/ontology/audit (00) → RQs/สมมติฐาน/โปรโตคอลคงที่ → canonical baseline table → controlled ablations → uncertainty/error/boundary/slices → efficiency จริง → ข้อจำกัดและช่องว่าง deployment การอ้าง novelty ต้องมี contribution และเทียบวรรณกรรมตรง task การเพิ่ม plots อย่างเดียวไม่สร้าง novelty

In [19]:
if EFFICIENCY and len(TABLE):
    measured=pd.DataFrame(EFFICIENCY)
    primary=TABLE[TABLE.seed==SEEDS[0]].merge(measured,on='model',validate='one_to_one')
    for _,r in primary.iterrows():
        assert r.checkpoint_sha==file_sha(CHECKPOINTS[(r.model,SEEDS[0])]), 'Trade-off checkpoint differs'
    primary.to_csv(OUT/'accuracy_efficiency.csv',index=False)
    fig,axes=plt.subplots(1,2,figsize=(11,4))
    for ax,metric,title in zip(axes,['box_AP','mask_AP'],['Box benchmark','Mask benchmark']):
        for _,r in primary.dropna(subset=[metric]).iterrows():
            ax.scatter(r.median_ms,r[metric]); ax.annotate(r.model,(r.median_ms,r[metric]),fontsize=7)
            ax.plot([r.median_ms,r.p95_ms],[r[metric],r[metric]],color='gray',alpha=.5)
        ax.set(xlabel='Measured median → p95 latency (ms)',ylabel='Canonical AP',ylim=(0,1),title=title+'; native recipes')
    savefig('accuracy_efficiency')
else: print('Trade-off pending matching measured latency and verified results; no estimated points plotted.')
write_json(OUT/'execution_summary.json',dict(dataset_id=DATA_ID,code_id=CODE_ID,flags=dict(train=DO_TRAIN,infer=DO_INFER,test=DO_EVAL_TEST,bootstrap=DO_BOOTSTRAP,ablations=DO_ABLATIONS,benchmark=DO_BENCHMARK,export=DO_EXPORT),checkpoints=len(CHECKPOINTS),validation_runs=len(VAL_PREDS),test_runs=len(TEST_PREDS),headline='Research workflow only; publication and production evidence incomplete'))

Trade-off pending matching measured latency and verified results; no estimated points plotted.


## 13. เอกสารอ้างอิงหลักและการประเมินงาน

**สถานะ:** ใช้เป็น workflow วิจัยที่ตรวจย้อนกลับได้เมื่อรัน gated experiments แล้ว ยังไม่ใช่ชุดผลทดลองวิทยานิพนธ์ที่เสร็จหรือระบบ production การแก้โค้ดไม่ทำให้ metrics เก่าของ seen-scenes กลายเป็นหลักฐานอิสระย้อนหลัง

| เอกสาร | ใช้ในงานนี้ | ขอบเขต implementation |
|---|---|---|
| [COCO detection evaluation](https://cocodataset.org/dataset/detection-eval.htm) | AP50:95/AP50/AP75, size/recall และ official mask/box evaluator | Inline reference ใช้วิเคราะห์; คะแนนหลักใช้ COCOeval |
| [Cheng et al., Boundary IoU, CVPR 2021](https://arxiv.org/abs/2103.16562) | Mask IoU ของบริเวณใหญ่อาจซ่อน boundary errors | Raw Boundary IoU ของ matched instances ไม่ใช่ Boundary AP; ต้องดู interior errors ด้วย |
| [Bolya et al., TIDE, ECCV 2020](https://www.ecva.net/papers/eccv_2020/papers_ECCV/papers/123480562.pdf) | แยก classification/localization/duplicate/background/missed errors | จำนวนเชิงบรรยายไม่ใช่ canonical oracle dAP |
| [REFORMS, Science Advances](https://www.science.org/doi/10.1126/sciadv.adk3452) | Data dependencies, preprocessing leakage, reproducibility, uncertainty และขอบเขตผล | Group identity ไม่พิสูจน์ event/site independence |
| [NIST AI RMF MEASURE](https://airc.nist.gov/airmf-resources/playbook/measure/) | บันทึก test sets, metrics, monitoring และหลักฐานที่ขาด | แนวทางสมัครใจ ไม่ใช่ certification/deployment approval |

ต้นฉบับบันทึกการทบทวนออนไลน์วันที่ 2026-10-13 เป็นการทบทวนเฉพาะประเด็น ไม่ใช่ SOTA survey ครบ ต้องเพิ่มงาน material-region segmentation ที่ ontology/label/acquisition protocol เทียบกันได้ก่อนอ้าง novelty หรือ SOTA

**ทำแล้ว:** แก้ syntax/validation ใน 00, สร้างและ audit fold-local offline aug, ออก handoff ใหม่, ต่อ CV receiver และจัด dependencies แล้ว GPU smoke ทั้งสอง arms ผ่าน พร้อมเก็บ outputs ใน notebook **เหลือก่อน full CV train:** implement matched-step CV runner และอนุมัติ paired full-run budget ก่อนฝึก/ประเมิน outer-heldout

**งานหลักฐานหลังจากนั้น:** complete seeds/ablations และ uncertainty; human labels/inter-annotator review; negative scenes; target-runtime numerical/task parity และ hardware/energy benchmarks; ผู้รับผิดชอบ monitoring/acceptance/rollback เหตุการณ์/ไซต์ใหม่เลื่อนไว้ ไม่ต้องสร้าง untouched confirmatory test จากข้อมูลเดิมเพื่อเริ่ม train แต่ยังอ้าง event/site generalization ไม่ได้

## 14. CV preflight v2 — 5 folds × 2 data arms

โปรโตคอล v2 มี **2 data arms** (`control`, `offline_aug`) ใช้ source parents, group-disjoint roles และ inner-valid/outer-heldout เดียวกัน 5 folds ทุกภาพเป็น outer-heldout หนึ่งครั้ง ตัวรับ handoff รับเฉพาะ `nb00-groupcv-v2` ที่ใช้ variants `color_light / clahe_blur / environment` และตรวจ hashes ทุกไฟล์โดยไม่ทำ prediction บน outer-heldout

Output ตารางด้านล่างเป็น **หลักฐาน historical v1** จาก `groupcv_offline_v1_253fef93327e4d8a` ยังไม่ถูกลบตามข้อตกลงเก็บประวัติ แต่โค้ดใหม่จะไม่ใช้ v1 แทนสูตร v2

คู่ทดลองหลักของ v2 คือ **YOLO26n-seg และ YOLO26s-seg** เพื่อตอบว่า offline + online augmentation recipe ทั้งชุดช่วยหรือไม่ ไม่พิสูจน์ผลของ transform รายตัว Benchmark 6-model เดิมและ fixed-split ablations เดิมเก็บแยกเป็นบริบทเดิม

Preflight ของ v2 ทำในส่วน 15.2: preview จาก production loader และ bounded production-batch smoke โดยยังไม่เปิด full training หรือ evaluation จนเจ้าของอนุมัติ protocol/preview identity

In [20]:
CV_ARMS = ['control','offline_aug']
DO_CV_SMOKE = os.environ.get('NB16_CV_SMOKE','0') == '1'
CV_HANDOFF = None
CV_ROOT = None
CV_LINEAGE = None
CV_PREFLIGHT = []
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')


def check_cv_group_roles(frame):
    assert frame.groupby('cv_group').role.nunique().eq(1).all(), 'CV group crosses roles'


def load_cv_handoff():
    source=MANIFEST.set_index('stem')
    for path in sorted((EXP/'results/nb00_audit').glob('*/cv_handoff.json'),reverse=True):
        item=json.loads(path.read_text())
        if item.get('schema')!='nb00-groupcv-v2' or item.get('source_dataset_id')!=DATA_ID or item.get('source_code_id')!=nb00_source: continue
        assert item['status']=='PASS' and item['geometry_status']=='shapely validity checked'
        root=Path(item['cv_root']).resolve()
        assert root.parent==(EXP/'data').resolve() and root.name.startswith('groupcv_offline_v2_')
        manifest_path=root/'manifest.json'
        assert file_sha(manifest_path)==item['manifest_sha']
        manifest=json.loads(manifest_path.read_text())
        assert manifest['spec']==item['protocol'] and manifest['cv_id']==item['cv_id']
        assert hashlib.sha256(json.dumps(manifest['spec'],sort_keys=True).encode()).hexdigest()==item['cv_id']
        assert manifest['spec']['schema']=='groupcv-offline-v2' and item['folds']==manifest['spec']['k']==5
        assert manifest['spec']['augmentation']['variants']==['color_light','clahe_blur','environment']
        assert manifest['spec']['sampling']==dict(parent='uniform_with_replacement',aug_variant_probabilities={v:.25 for v in ['original','color_light','clahe_blur','environment']},control_variant='original',hard_scene_oversampling=False)
        for name,sha in manifest['artifacts'].items():
            artifact=(root/name).resolve()
            assert artifact.is_relative_to(root) and file_sha(artifact)==sha
        assignments=pd.read_csv(root/'assignments.csv',keep_default_na=False)
        assert set(assignments.fold)==set(range(item['folds']))
        assert hashlib.sha256(assignments.sort_values(['fold','stem']).to_csv(index=False).encode()).hexdigest()==manifest['spec']['assignments_sha']
        held=assignments[assignments.role=='heldout']
        assert set(held.stem)==set(source.index) and held.stem.value_counts().eq(1).all()
        lineage=pd.read_csv(root/'lineage.csv',keep_default_na=False)
        assert set(lineage.arm)==set(CV_ARMS) and set(lineage.fold)==set(range(item['folds'])) and not lineage.duplicated(['fold','arm','image']).any()
        reports=[]
        for fold in range(item['folds']):
            frame=assignments[assignments.fold==fold]
            assert frame.stem.is_unique and set(frame.stem)==set(source.index) and set(frame.role)=={'train','valid','heldout'}
            check_cv_group_roles(frame)
            assert frame.groupby('original_group').role.nunique().eq(1).all()
            assert frame.groupby('role')[['gravel','sand']].sum().gt(0).all().all()
            for field in ['upload_uuid','image_sha','decoded_sha']:
                subset=frame[frame[field]!='']; assert subset.groupby(field).role.nunique().eq(1).all()
            for field in ['image_sha','label_sha','decoded_sha','gravel','sand']:
                assert frame.set_index('stem')[field].sort_index().equals(source[field].sort_index())
            assert frame.set_index('stem').original_group.to_dict()==source.group.to_dict()
            reference=json.loads((root/f'quality_reference_fold{fold}.json').read_text())
            assert reference['selected_on']=='inner_train' and set(reference['source_stems'])==set(frame.loc[frame.role=='train','stem'])
            assignment=frame.set_index('stem')
            for arm in CV_ARMS:
                sub=lineage[(lineage.fold==fold)&(lineage.arm==arm)]
                assert sub.groupby('cv_group').role.nunique().eq(1).all()
                for role,split in [('train','train'),('valid','valid'),('heldout','test')]:
                    rows=sub[sub.role==role]; expected=set(frame.loc[frame.role==role,'stem'])
                    variants={'original','color_light','clahe_blur','environment'} if arm=='offline_aug' and role=='train' else {'original'}
                    assert set(rows.parent_stem)==expected and len(rows)==len(expected)*len(variants)
                    assert rows.groupby('parent_stem').variant.apply(set).map(lambda x:x==variants).all()
                    directory=root/f'fold{fold}'/arm/split
                    assert set(p.name for p in (directory/'images').iterdir())==set(rows.image)
                    assert set(p.stem for p in (directory/'labels').glob('*.txt'))==set(rows.stem)
                    for r in rows.itertuples(index=False):
                        original=source.loc[r.parent_stem]; owner=assignment.loc[r.parent_stem]
                        assert r.role==owner.role and r.cv_group==owner.cv_group and r.original_group==owner.original_group
                        assert r.source_image_sha==original.image_sha and r.source_label_sha==original.label_sha
                        for name,sha,kind in [(r.image_path,r.image_sha,'images'),(r.label_path,r.label_sha,'labels')]:
                            artifact=(root/name).resolve()
                            assert artifact.parent==(directory/kind).resolve() and file_sha(artifact)==sha
                        assert r.label_sha==original.label_sha
                        if r.variant=='original': assert r.image_sha==original.image_sha
                    reports.append(dict(fold=fold,arm=arm,role=role,images=len(rows),parents=len(expected),status='PASS'))
        return item,root,lineage,reports
    return None,None,None,[]


CV_HANDOFF,CV_ROOT,CV_LINEAGE,CV_PREFLIGHT=load_cv_handoff()
if CV_HANDOFF is None:
    print('PENDING v2: run 00 audit/materialization; v1 handoff is intentionally not accepted.')
    if DO_CV_SMOKE: raise RuntimeError('CV smoke blocked: missing/stale v2 handoff')
else:
    write_json(OUT/'cv_handoff_used.json',CV_HANDOFF)
    pd.DataFrame(CV_PREFLIGHT).to_csv(OUT/'cv_preflight.csv',index=False)
    print('PASS audited v2 handoff:',CV_ROOT); display(pd.DataFrame(CV_PREFLIGHT))
try:
    check_cv_group_roles(pd.DataFrame({'cv_group':['same','same'],'role':['train','heldout']}))
except AssertionError: print('PASS regression: cross-role group rejected')
else: raise AssertionError('Leakage regression did not reject invalid cohort')

PASS audited v2 handoff: /home/supawich/Desktop/Aggregate_Project/experiment/data/groupcv_offline_v2_de6276f231115417


,fold,arm,role,images,parents,status
0,0,control,train,737,737,PASS
1,0,control,valid,183,183,PASS
2,0,control,heldout,164,164,PASS
3,0,offline_aug,train,2948,737,PASS
4,0,offline_aug,valid,183,183,PASS
5,0,offline_aug,heldout,164,164,PASS
6,1,control,train,543,543,PASS
7,1,control,valid,140,140,PASS
8,1,control,heldout,401,401,PASS
9,1,offline_aug,train,2172,543,PASS


PASS regression: cross-role group rejected


### 14.1 Helper สำหรับเลือก validation parents ใน smoke

เก็บ helper เลือก validation parents ที่มีทั้งสองคลาสไว้ใน cell ถัดไป ส่วน bounded GPU smoke ของ v2 ที่ใช้ production batch และ online/offline pipeline จริง ย้ายไปอยู่ใน cell 43 เพื่อไม่ให้เปิด `NB16_CV_SMOKE=1` แล้วรันสองครั้งซ้ำกัน

In [21]:
CV_SMOKE_RESULTS = []
SMOKE_MAX_STEPS = 2


def smoke_parents(frame, role, limit=4):
    rows = frame[(frame.role==role)&(frame.variant=='original')].sort_values('parent_stem')
    selected = []
    for cls in CLS:
        for stem in rows.parent_stem:
            if MANIFEST.set_index('stem').loc[stem,cls] > 0 and stem not in selected:
                selected.append(stem)
                if sum(MANIFEST.set_index('stem').loc[s,cls] > 0 for s in selected) >= 2: break
    for stem in rows.parent_stem:
        if len(selected) >= limit: break
        if stem not in selected: selected.append(stem)
    assert len(selected) <= limit and MANIFEST.set_index('stem').loc[selected,CLS].sum().gt(0).all()
    return selected


print('Legacy batch-2 smoke helper retained. Production-batch v2 smoke executes in cell 43 when NB16_CV_SMOKE=1.')

Legacy batch-2 smoke helper retained. Production-batch v2 smoke executes in cell 43 when NB16_CV_SMOKE=1.


## บันทึกส่งต่องาน — Protocol v2 (2026-10-14)

### สถานะปัจจุบัน
- **โค้ดอัปเดตเป็น v2 แล้ว:** `00` ใช้ offline train variants `color_light`, `clahe_blur`, `environment`; `16` ใช้ parent-uniform → variant sampling, online augmentation เฉพาะ `offline_aug`, usage counters, state-verified resume, freeze record และแยก train/freeze/evaluate
- **ยังไม่สร้าง dataset v2 และยังไม่เริ่ม train/evaluate:** ค่าเริ่มต้นเป็น audit-only ทุก action
- **ประวัติ v1 เก็บแยก ไม่ปะปนกับ v2:** `data/groupcv_offline_v1_253fef93327e4d8a` และ `results/nb16_full_cv/*` เป็นของสูตรเก่า (offline `hflip/gamma_brightness/clahe_blur`, online ปิดทั้งหมด; มีหนึ่งชุดจบที่ 43 completed / 17 failed จาก 60 งาน) ตัวรับ v2 ปฏิเสธ handoff v1 โดยอัตโนมัติ

### ขอบเขตและการตีความ
- **คำถามวิจัย:** ทดสอบว่า augmentation recipe ทั้งชุด (offline + online) ช่วย **YOLO26n-seg / YOLO26s-seg** หรือไม่ ภายใต้ parent-uniform sampling และงบ optimizer updates เท่ากันภายในคู่ ไม่ใช่พิสูจน์ผลของ transform รายตัว
- **Split หลัก:** grouped development CV 5 folds จาก `gold_fixed_v1` (fixed split เดิมคือ 753/226/105)
- **Stripe 88 ภาพ:** ต้องติดป้าย **development-inspected, seen-scene test** ไม่เรียก untouched และไม่ใช่ CV outer-heldout
- **External ZIP 3 ชุด:** ยังไม่เรียก independent holdout จนผ่าน mechanical audit + human ontology/annotation/duplicate review และรายงานแยกชุดเสมอ

### ลำดับตรวจและรันเมื่อเจ้าของอนุมัติ
1. รัน `00` cells **1, 66, 68, 70, 72, 74, 76** เพื่อดู preview ของ offline variants จาก pipeline จริง
2. เมื่อยืนยันค่า offline แล้ว ตั้ง `NB00_BUILD_CV=1` แล้วรัน cells **78, 80** เพื่อสร้างและ audit `groupcv_offline_v2_<id>` พร้อม `cv_handoff.json`
3. รัน `16` audit mode แล้วเปิดทีละขั้น:
   - `NB16_CV_PREVIEW=1` → ตรวจ preview และ mask alignment จาก production loader
   - `NB16_CV_SMOKE=1` → รัน bounded production-batch smoke (2 steps ต่อ arm/model, outer predictions = 0)
   - ตั้ง `NB16_APPROVED_PROTOCOL` และ `NB16_APPROVED_PREVIEW_SHA` ตามค่าที่พิมพ์ออกมา
   - `NB16_FULL_CV=1` → ฝึกครบตามงบแล้วหยุดก่อน heldout
   - `NB16_CV_FREEZE=1` → ล็อก checkpoint + inner-valid thresholds ลง `freeze.json`
   - `NB16_CV_EVALUATE=1` → ประเมิน outer-heldout จากโมเดลและ thresholds ที่ freeze แล้วเท่านั้น
   - `NB16_EXTERNAL_AUDIT=1` → ตรวจ external ZIPs ก่อนกรอก `EXTERNAL_REVIEWS` และค่อยเปิด `NB16_EVAL_EXTERNAL=1` หรือ `NB16_EVAL_STRIPE=1` แยกต่างหาก

## 15. Protocol v2 — Parent-uniform sampling และ offline + online augmentation

**ขอบเขตของส่วนนี้:** ทดสอบว่า augmentation recipe ทั้งชุด (offline + online) ช่วย **YOLO26n-seg และ YOLO26s-seg** หรือไม่ ภายใต้ source split เดียวกัน, parent-uniform sampling และ optimizer updates เท่ากันภายในคู่ (2 models × 2 data arms × 5 folds × seed 42 = **20 runs** สำหรับงบเริ่มต้น)

### เจ้าของ augmentation และ sampling
- **`00` (offline, เฉพาะ inner-train):** `original`, `color_light` (gamma `0.85–1.90`, brightness `0.90–1.10`, hue `±15°` p=0.5, saturation `0.85–1.15` p=0.5, white balance RGB gains `0.95–1.05` p=0.5), `clahe_blur` (CLAHE clip=2, grid `8×8`, Gaussian blur sigma=0.35 p=0.5), `environment` (เลือก rain / noise / night อย่างใดอย่างหนึ่งด้วยโอกาส `1/3`)
- **Sampling ใน `16`:** สุ่ม parent อย่างเท่าเทียมแบบ with replacement → `control` ใช้ `original` 100%; `offline_aug` สุ่ม `original / color_light / clahe_blur / environment` อย่างละ 25% แยก primary-parent RNG จาก donor RNG เพื่อให้ลำดับ primary parent ของ `control` และ `offline_aug` ตรงกัน ไม่มี hard-scene oversampling
- **`16` (online, เฉพาะ `offline_aug`):** horizontal flip `p=0.5`, vertical flip ปิด, affine (rotation `±10°`, translate `±5%`, scale `0.9–1.1`, shear `±3°`), crop เกิดจากการจัดวาง/clipping ของ affine และ Mosaic เท่านั้น, Mosaic `p=0.25`, Copy-Paste `p=0.10` (`mode='mixup'` พร้อม donor จาก inner-train ที่ผ่าน parent → variant sampling เดียวกัน)
- **ปิดทั้งสอง arms:** MixUp, CutMix, online HSV/color jitter และ Albumentations color transforms เพื่อไม่ให้ซ้ำกับ offline recipe
- **Valid / outer-heldout / Stripe / external:** ไม่ทำ offline หรือ online augmentation ใช้เฉพาะ evaluation preprocessing มาตรฐาน

### การควบคุมงบ, การบันทึก และการแยกขั้นตอน
- แต่ละ fold/model มี virtual epoch ยาว `ceil(N_train_parents / batch)` steps และใช้ `drop_last=True` กับ dataset ความยาว `steps * batch` เพื่อให้ทั้งคู่ประมวลผลจำนวน output images และ optimizer updates เท่ากันทุก epoch (`yolo26n_seg=12`, `yolo26s_seg=6`, 30 virtual epochs)
- บันทึกแยก: `optimizer_steps`, `output_images`, `source_parent_draws`, `primary_parent_draws`, `donor_parent_draws`, `unique_source_files`, `unique_parents`, `variant_draws`, ตาราง `sampling_counts.csv` ราย `(parent, variant)`, และเวลาฝึกแยกจาก validation/evaluation
- แยกเป็น 3 ขั้นตอนที่เปิดพร้อมกันไม่ได้: `NB16_FULL_CV=1` (train + เลือก checkpoint จาก inner-valid) → `NB16_CV_FREEZE=1` (ล็อก checkpoint hash + inner-valid thresholds ลง `freeze.json`) → `NB16_CV_EVALUATE=1` (ประเมิน outer-heldout จาก `freeze.json` เท่านั้น)
- Outputs ที่ติดอยู่ใต้ cell 39 เดิมเป็น log ของการรัน v1 เก่า ไม่ใช่ผลของโค้ด v2 นี้

In [22]:
import math, gc, copy, gzip
from collections import Counter

DO_FULL_CV = os.environ.get('NB16_FULL_CV','0') == '1'
DO_CV_PREVIEW = os.environ.get('NB16_CV_PREVIEW','0') == '1'
DO_CV_FREEZE = os.environ.get('NB16_CV_FREEZE','0') == '1'
DO_CV_EVALUATE = os.environ.get('NB16_CV_EVALUATE','0') == '1'
CV_APPROVED_PROTOCOL = os.environ.get('NB16_APPROVED_PROTOCOL','')
CV_APPROVED_PREVIEW_SHA = os.environ.get('NB16_APPROVED_PREVIEW_SHA','')
CV_FULL_SEEDS = [42]
CV_FULL_EPOCHS = 30
CV_BATCH_SIZES = {'yolo26n_seg':12,'yolo26s_seg':6}
CV_FULL_RECIPES = [dict(r,epochs=CV_FULL_EPOCHS,batch=CV_BATCH_SIZES[r['name']]) for r in RECIPES if r['name'] in CV_BATCH_SIZES]
CV_VARIANTS = ['original','color_light','clahe_blur','environment']
CV_ONLINE = dict(fliplr=.5,flipud=0.,degrees=10.,translate=.05,scale=.1,shear=3.,perspective=0.,mosaic=.25,copy_paste=.10,copy_paste_mode='mixup',copy_paste_fraction=1.,close_mosaic=0,mixup=0.,cutmix=0.,hsv_h=0.,hsv_s=0.,hsv_v=0.,bgr=0.,crop='affine clipping only',albumentations=False)
CV_TRAIN_ARGS = dict(optimizer='AdamW',lr0=.001,lrf=.01,cos_lr=False,momentum=.937,weight_decay=.0005,warmup_epochs=0,amp=False,deterministic=True,workers=0,cache=False,plots=False,save=True,overlap_mask=True,mask_ratio=4,channels_last=False,compile=False)


def cv_json_id(value):
    return hashlib.sha256(json.dumps(value,sort_keys=True,allow_nan=False).encode()).hexdigest()


def cv_atomic_json(path,value):
    path=Path(path); temporary=path.with_name(path.name+'.tmp')
    write_json(temporary,value); os.replace(temporary,path)


def cv_atomic_torch(path,value):
    torch,_=runtime(); path=Path(path); temporary=path.with_name(path.name+'.tmp')
    torch.save(value,temporary); os.replace(temporary,path)


class CVParentSampler:
    def __init__(self,rows,seed,augmented):
        assert not rows.empty and rows.role.eq('train').all() and not rows.duplicated(['parent_stem','variant']).any()
        self.parents=sorted(rows.parent_stem.unique())
        self.variants=CV_VARIANTS if augmented else ['original']
        assert rows.groupby('parent_stem').variant.apply(set).map(lambda x:x==set(self.variants)).all()
        self.files={(r.parent_stem,r.variant):r.image for r in rows.itertuples(index=False)}
        self.parent_rng=np.random.default_rng(seed)
        self.donor_rng=np.random.default_rng(seed+1)
        self.variant_rng=np.random.default_rng(seed+2)
        self.counts=Counter(); self.primary_counts=Counter(); self.outputs=0; self.draws=[]
    def draw(self,primary=False):
        rng=self.parent_rng if primary else self.donor_rng
        parent=self.parents[int(rng.integers(len(self.parents)))]
        variant=self.variants[int(self.variant_rng.integers(len(self.variants)))] if len(self.variants)>1 else 'original'
        key=(parent,variant); self.counts[key]+=1
        if primary: self.primary_counts[key]+=1
        self.draws.append(dict(parent=parent,variant=variant,primary=primary))
        return self.files[key]
    def state_dict(self):
        return dict(parent_rng=copy.deepcopy(self.parent_rng.bit_generator.state),donor_rng=copy.deepcopy(self.donor_rng.bit_generator.state),variant_rng=copy.deepcopy(self.variant_rng.bit_generator.state),counts=dict(self.counts),primary_counts=dict(self.primary_counts),outputs=self.outputs)
    def load_state_dict(self,state):
        for name in ['parent_rng','donor_rng','variant_rng']:
            getattr(self,name).bit_generator.state=copy.deepcopy(state[name])
        self.counts=Counter(state['counts']); self.primary_counts=Counter(state['primary_counts']); self.outputs=state['outputs']; self.draws=[]
    def summary(self):
        return dict(output_images=self.outputs,source_parent_draws=sum(self.counts.values()),primary_parent_draws=sum(self.primary_counts.values()),donor_parent_draws=sum(self.counts.values())-sum(self.primary_counts.values()),unique_source_files=len(self.counts),unique_parents=len({p for p,v in self.counts}),variant_draws={v:sum(n for (p,k),n in self.counts.items() if k==v) for v in self.variants})
    def table(self):
        return pd.DataFrame([dict(parent_stem=p,variant=v,source_draws=self.counts[(p,v)],primary_draws=self.primary_counts[(p,v)],donor_draws=self.counts[(p,v)]-self.primary_counts[(p,v)]) for p in self.parents for v in self.variants])


def cv_parent_dataset(args,data,root,rows,seed,batch,steps,augmented):
    from ultralytics.data.dataset import YOLODataset
    from ultralytics.data.augment import Compose,LetterBox,Mosaic,RandomPerspective,RandomFlip,CopyPaste
    sampler=CVParentSampler(rows,seed,augmented)
    config=dict(CV_ONLINE) if augmented else dict(CV_ONLINE,fliplr=0.,degrees=0.,translate=0.,scale=0.,shear=0.,mosaic=0.,copy_paste=0.)

    class ParentDataset(YOLODataset):
        def __len__(self): return steps*batch if hasattr(self,'parent_sampler') else super().__len__()
        def get_image_and_label(self,index,primary=False):
            name=self.parent_sampler.draw(primary)
            return super().get_image_and_label(self.file_indexes[name])
        def __getitem__(self,index):
            self.parent_sampler.draws=[]
            result=self.transforms(self.get_image_and_label(index,primary=True))
            assert result['img'].shape==(3,args.imgsz,args.imgsz)
            assert len(result['cls'])==len(result['bboxes']) and result['masks'].ndim==3
            assert np.isfinite(result['bboxes'].numpy()).all() and np.isfinite(result['masks'].numpy()).all()
            self.parent_sampler.outputs+=1
            return result

    dataset=ParentDataset(img_path=str(Path(root)/'train/images'),imgsz=args.imgsz,batch_size=batch,augment=True,hyp=copy.copy(args),rect=False,cache=False,single_cls=False,stride=32,pad=0.,task='segment',data=data,fraction=1.)
    dataset.file_indexes={Path(name).name:i for i,name in enumerate(dataset.im_files)}
    assert set(dataset.file_indexes)==set(rows.image), 'Loader dropped/replaced files; audit labels first'
    lookup=rows.set_index('image')
    for label in dataset.labels:
        assert len(label['cls'])==int(lookup.loc[Path(label['im_file']).name,'instances']), 'Loader changed annotation count'
    dataset.parent_sampler=sampler
    formatter=dataset.transforms.transforms[-1]

    class ParentMosaic(Mosaic):
        def get_indexes(self): return [0,0,0]

    class DonorCopyPaste(CopyPaste):
        def get_indexes(self): return 0
        def __call__(self,labels):
            if random.random()>=config['copy_paste']: return labels
            return super().__call__(labels)

    affine=lambda: RandomPerspective(degrees=config['degrees'],translate=config['translate'],scale=config['scale'],shear=config['shear'],perspective=0.,size=(args.imgsz,args.imgsz))
    if augmented:
        dataset.transforms=Compose([ParentMosaic(dataset,imgsz=args.imgsz,p=config['mosaic']),affine(),DonorCopyPaste(dataset,pre_transform=Compose([affine()]),p=config['copy_paste_fraction'],mode='mixup'),RandomFlip(direction='horizontal',p=config['fliplr']),formatter])
    else: dataset.transforms=Compose([LetterBox(new_shape=(args.imgsz,args.imgsz),scaleup=True),formatter])
    dataset.pipeline_config=config
    return dataset


def cv_training_rows(fold,arm):
    rows=CV_LINEAGE[(CV_LINEAGE.fold==fold)&(CV_LINEAGE.arm==arm)&(CV_LINEAGE.role=='train')].copy()
    source=MANIFEST.set_index('stem')
    rows['instances']=rows.parent_stem.map((source.gravel+source.sand).to_dict())
    assert rows.instances.notna().all()
    return rows


def cv_plan():
    assert CV_HANDOFF is not None and len(CV_PREFLIGHT)==30, 'Build/audit v2 in 00 first'
    assert CV_HANDOFF['schema']=='nb00-groupcv-v2'
    assert CV_HANDOFF['protocol']['augmentation']['variants']==CV_VARIANTS[1:]
    assert CV_HANDOFF['protocol']['sampling']==dict(parent='uniform_with_replacement',aug_variant_probabilities={v:.25 for v in CV_VARIANTS},control_variant='original',hard_scene_oversampling=False)
    packages={p:importlib.metadata.version(p) for p in ['ultralytics','torch','torchvision','numpy','opencv-python','pycocotools']}
    assert packages['ultralytics']=='8.4.171', 'Review loader/resume adapter before changing the verified Ultralytics API'
    jobs=[dict(recipe=r,fold=f,seed=s,arm=a) for f in range(5) for r in CV_FULL_RECIPES for s in CV_FULL_SEEDS for a in CV_ARMS]
    plan=dict(schema='nb16-parent-recipe-v2',cv_id=CV_HANDOFF['cv_id'],dataset_id=DATA_ID,source_id=CODE_ID,packages=packages,jobs=jobs,offline=CV_HANDOFF['protocol']['augmentation'],online=CV_ONLINE,training=CV_TRAIN_ARGS,sampling=CV_HANDOFF['protocol']['sampling'],selection='inner-valid canonical mask AP; thresholds inner-valid; freeze before evaluation',scope='development grouped CV; one seed; recipe-level effect, not transform causality',resume='last completed virtual epoch; raw FP32 weights, optimizer, scheduler, scaler, EMA, criterion, RNG and parent sampler')
    return plan,cv_json_id(plan)[:20]

### 15.1 Training state, freeze และ evaluation แยกขั้นตอน

Runner v2 ใช้ native Ultralytics trainer/loss/optimizer กับ production loader ของ cell 39 ไม่เปิดสีผ่าน Albumentations หรือ HSV เพิ่ม Both arms ได้ full batch และ updates เท่ากัน แยก primary-parent RNG จาก donor RNG เพื่อให้ primary parent sequence ภายในคู่เหมือนกัน

Resume จาก **ขอบเขต virtual epoch ที่บันทึกสำเร็จล่าสุด** เท่านั้น เก็บ raw FP32 model, optimizer, scheduler, scaler, EMA, criterion schedule, Python/NumPy/Torch/CUDA RNG, sampler RNG/counters และ selection curve พร้อม spec/checksum ไม่ใช้ selected checkpoint ฝึกต่อ ไม่เริ่มใหม่เงียบ ๆ หาก state หาย งานที่หยุดกลาง epoch replay จากขอบเขตล่าสุด; logs ของ epoch ที่ไม่ commit ไม่ถือเป็น completed budget

เวลาแยก `training_seconds` (รวม loader แต่ไม่รวม validation), `gpu_elapsed_seconds` (CUDA events), และ GPU-synchronized batch wall time จำนวน output images / source draws / parent-variant usage / unique files เก็บตามจริง รวม donor/composite inputs แม้ instance ถูก clipping ออก

Checkpoint เลือกจาก inner-valid canonical mask AP ที่ matched steps → เปิด freeze stage เพื่อเลือก threshold จาก inner-valid แล้วบันทึก `freeze.json` → เปิด evaluation stage คนละครั้ง ห้าม train/freeze/evaluate พร้อมกัน ไม่มีการเลือก threshold ใหม่จาก heldout

In [23]:
def cv_preview(plan,token):
    from ultralytics.cfg import get_cfg
    samples=[]; fig,axes=plt.subplots(2,4,figsize=(16,8))
    for row,arm in enumerate(CV_ARMS):
        seed_everything(42)
        args=get_cfg(overrides=dict(imgsz=960,task='segment',mask_ratio=CV_TRAIN_ARGS['mask_ratio'],overlap_mask=CV_TRAIN_ARGS['overlap_mask']))
        ds=cv_parent_dataset(args,{'names':{0:'gravel',1:'sand'}},CV_ROOT/'fold0'/arm,cv_training_rows(0,arm),42,2,8,arm=='offline_aug')
        for index in range(16):
            sample=ds[index]
            samples.append(dict(arm=arm,index=index,sources=copy.deepcopy(ds.parent_sampler.draws),image_sha=hashlib.sha256(sample['img'].numpy().tobytes()).hexdigest(),mask_sha=hashlib.sha256(sample['masks'].numpy().tobytes()).hexdigest()))
            if index<4:
                image=sample['img'].permute(1,2,0).numpy(); masks=sample['masks'].numpy()
                foreground=cv2.resize((masks.max(axis=0)>0).astype(np.uint8),(960,960),interpolation=cv2.INTER_NEAREST) if len(masks) else np.zeros((960,960),np.uint8)
                axes[row,index].imshow(image)
                if foreground.any(): axes[row,index].contour(foreground,levels=[.5],colors=['cyan'],linewidths=.6)
                axes[row,index].set_title(f'{arm}: {len(ds.parent_sampler.draws)} sources'); axes[row,index].axis('off')
    path=OUT/('cv_preview_'+token+'.json')
    write_json(path,dict(protocol_id=token,plan=plan,samples=samples,alignment_review='PENDING human review',training_executed=False))
    plt.tight_layout(); savefig('cv_production_preview_'+token)
    print('Review offline/online masks and realism. Approval IDs:',token,file_sha(path))
    return path


def cv_resume_load(path,spec):
    torch,_=runtime(); meta=Path(path).with_suffix('.json')
    assert meta.is_file(), 'Missing resume identity/checksum'
    stored=json.loads(meta.read_text())
    assert stored['spec']==spec and stored['checkpoint_sha']==file_sha(path), 'Resume identity/checksum differs'
    checkpoint=torch.load(path,map_location='cpu',weights_only=False)
    assert checkpoint['protocol_spec']==spec and checkpoint['epoch']>=0
    assert all(k in checkpoint for k in ['model','ema','optimizer','scheduler','scaler','rng','sampler','cv_steps','selection','timing','criterion_state','training_init_sha','selected_sha'])
    assert checkpoint['cv_steps']==(checkpoint['epoch']+1)*spec['steps_per_epoch']
    assert checkpoint['sampler']['outputs']==checkpoint['cv_steps']*spec['job']['recipe']['batch']
    assert checkpoint['selected_sha']==file_sha(Path(path).parent/'selected.pt'), 'Selected checkpoint differs from committed training state'
    return checkpoint


def cv_parameter_sha(model):
    digest=hashlib.sha256()
    for name,p in model.named_parameters(): digest.update(name.encode()); digest.update(p.detach().cpu().contiguous().numpy().tobytes())
    return digest.hexdigest()


def cv_train_yolo(recipe,seed,run_dir,root,valid,steps_per_epoch,spec,report=None):
    from ultralytics import YOLO
    from ultralytics.models.yolo.segment.train import SegmentationTrainer
    torch,device=seed_everything(seed)
    resume_path=run_dir/'resume.pt'
    checkpoint=cv_resume_load(resume_path,spec) if resume_path.exists() else None
    if checkpoint is not None and checkpoint['epoch']+1==recipe['epochs']:
        return run_dir/'selected.pt',checkpoint['timing'],checkpoint['sampler'],checkpoint['training_init_sha']
    assert checkpoint is not None or not (run_dir/'selected.pt').exists(), 'Partial state missing; choose a new run identity instead of restarting silently'
    rows=cv_training_rows(spec['job']['fold'],spec['job']['arm']); augmented=spec['job']['arm']=='offline_aug'

    class MatchedTrainer(SegmentationTrainer):
        def get_dataloader(self,dataset_path,batch_size=16,rank=0,mode='train'):
            if mode!='train': return super().get_dataloader(dataset_path,batch_size,rank,mode)
            assert rank in (-1,0) and self.world_size==1 and self.args.workers==0
            ds=cv_parent_dataset(self.args,self.data,root,rows,seed,batch_size,steps_per_epoch,augmented)
            return torch.utils.data.DataLoader(ds,batch_size=batch_size,shuffle=False,num_workers=0,collate_fn=ds.collate_fn,drop_last=True)
        def _setup_train(self):
            super()._setup_train()
            self.last=resume_path; self.best=run_dir/'selected.pt'
            self.cv_steps=0; self.cv_selection=[]
            self.cv_timing=dict(training_seconds=0.,gpu_synchronized_batch_seconds=0.,gpu_elapsed_seconds=0.)
            self.training_init_sha=cv_parameter_sha(self.model)
            if checkpoint is not None:
                self.model.load_state_dict(checkpoint['model'].float().state_dict())
                self.scheduler.load_state_dict(checkpoint['scheduler'])
                self.model.criterion=self.model.init_criterion()
                for key,value in checkpoint['criterion_state'].items():
                    assert hasattr(self.model.criterion,key); setattr(self.model.criterion,key,value)
                self.cv_steps=checkpoint['cv_steps']; self.cv_selection=checkpoint['selection']; self.cv_timing=checkpoint['timing']; self.training_init_sha=checkpoint['training_init_sha']
                self.train_loader.dataset.parent_sampler.load_state_dict(checkpoint['sampler'])
                random.setstate(checkpoint['rng']['python']); np.random.set_state(checkpoint['rng']['numpy'])
                torch.set_rng_state(checkpoint['rng']['torch']); torch.cuda.set_rng_state_all(checkpoint['rng']['cuda'])
            assert self.batch_size==recipe['batch'] and self.accumulate==1 and not self.amp
        def optimizer_step(self):
            assert self.batch_size==recipe['batch'] and self.accumulate==1, 'Runtime changed paired budget'
            assert torch.isfinite(self.loss).all().item(), 'Non-finite loss'
            gradients=[p.grad for p in self.model.parameters() if p.grad is not None]
            assert gradients and any(torch.count_nonzero(g).item() for g in gradients), 'No nonzero gradients'
            torch.nn.utils.clip_grad_norm_(self.model.parameters(),10.,error_if_nonfinite=True)
            super().optimizer_step(); self.cv_steps+=1
        def validate(self):
            proxy=YOLO(str(EXP/'weights'/recipe['init'])); proxy.model=copy.deepcopy(self.ema.ema).float().eval()
            predictions=predict_model(proxy,recipe,valid); validate_predictions(predictions,valid,'segment')
            score=canonical_coco(predictions,valid,'segm')['AP']
            assert score is not None and np.isfinite(score) and self.cv_steps==(self.epoch+1)*steps_per_epoch
            if self.best_fitness is None or score>self.best_fitness:
                cv_atomic_torch(self.best,dict(model=copy.deepcopy(proxy.model).cpu().float(),train_args=vars(self.args),epoch=self.epoch,version=importlib.metadata.version('ultralytics')))
                self.best_fitness=score
            record=dict(epoch=self.epoch+1,optimizer_steps=self.cv_steps,val_AP=score,loss=float(self.loss.detach()))
            self.cv_selection.append(record); pd.DataFrame(self.cv_selection).to_csv(run_dir/'selection.csv',index=False)
            if report: report(record)
            del predictions,proxy; gc.collect(); torch.cuda.empty_cache()
            return dict(cv_val_AP=score),score
        def save_model(self):
            assert self.cv_steps==(self.epoch+1)*steps_per_epoch
            payload=dict(epoch=self.epoch,best_fitness=self.best_fitness,model=copy.deepcopy(self.model).cpu().float(),ema=copy.deepcopy(self.ema.ema).cpu().float(),updates=self.ema.updates,optimizer=self.optimizer.state_dict(),scheduler=self.scheduler.state_dict(),scaler=self.scaler.state_dict(),train_args=vars(self.args),version=importlib.metadata.version('ultralytics'),protocol_spec=spec,cv_steps=self.cv_steps,selection=self.cv_selection,timing=self.cv_timing,training_init_sha=self.training_init_sha,training_final_sha=cv_parameter_sha(self.model),selected_sha=file_sha(self.best),criterion_state={k:getattr(self.model.criterion,k) for k in ['updates','o2m','o2o','o2m_copy','final_o2m','total'] if hasattr(self.model.criterion,k)},sampler=self.train_loader.dataset.parent_sampler.state_dict(),rng=dict(python=random.getstate(),numpy=np.random.get_state(),torch=torch.get_rng_state(),cuda=torch.cuda.get_rng_state_all()))
            payload['model'].criterion=None; payload['ema'].criterion=None
            cv_atomic_torch(self.last,payload)
            cv_atomic_json(self.last.with_suffix('.json'),dict(spec=spec,checkpoint_sha=file_sha(self.last)))
            self.train_loader.dataset.parent_sampler.table().to_csv(run_dir/'sampling_counts.csv',index=False)
            write_json(run_dir/'sampling_summary.json',self.train_loader.dataset.parent_sampler.summary())
            return True
        def final_eval(self): return None

    cfg=run_dir/'data.yaml'
    cfg.write_text(f'path: {Path(root).resolve()}\ntrain: train/images\nval: valid/images\nnames: {CLS}\n')
    model=YOLO(str(resume_path if checkpoint is not None else EXP/'weights'/recipe['init']))
    def epoch_start(trainer):
        torch.cuda.synchronize(); trainer.cv_epoch_start=time.perf_counter()
    def batch_start(trainer):
        torch.cuda.synchronize(); trainer.cv_batch_start=time.perf_counter()
        trainer.cv_gpu_start=torch.cuda.Event(enable_timing=True); trainer.cv_gpu_end=torch.cuda.Event(enable_timing=True)
        trainer.cv_gpu_start.record()
    def batch_end(trainer):
        trainer.cv_gpu_end.record(); torch.cuda.synchronize()
        trainer.cv_timing['gpu_elapsed_seconds']+=trainer.cv_gpu_start.elapsed_time(trainer.cv_gpu_end)/1000.
        trainer.cv_timing['gpu_synchronized_batch_seconds']+=time.perf_counter()-trainer.cv_batch_start
    def epoch_end(trainer):
        torch.cuda.synchronize(); trainer.cv_timing['training_seconds']+=time.perf_counter()-trainer.cv_epoch_start
    for event,callback in [('on_train_epoch_start',epoch_start),('on_train_batch_start',batch_start),('on_train_batch_end',batch_end),('on_train_epoch_end',epoch_end)]: model.add_callback(event,callback)
    online={k:v for k,v in CV_ONLINE.items() if k not in ['copy_paste_fraction','crop','albumentations']}
    if not augmented: online.update(fliplr=0.,degrees=0.,translate=0.,scale=0.,shear=0.,mosaic=0.,copy_paste=0.)
    model.train(trainer=MatchedTrainer,data=str(cfg),epochs=recipe['epochs'],patience=recipe['epochs']+1,batch=recipe['batch'],nbs=recipe['batch'],imgsz=recipe['size'],device=device,seed=seed,split='val',auto_augment=None,erasing=0,project=str(run_dir),name='native',exist_ok=True,resume=str(resume_path) if checkpoint is not None else False,**CV_TRAIN_ARGS,**online)
    trainer=model.trainer
    assert trainer.cv_steps==spec['optimizer_steps'] and trainer.train_loader.dataset.parent_sampler.outputs==trainer.cv_steps*recipe['batch']
    assert cv_parameter_sha(trainer.model)!=trainer.training_init_sha, 'No model parameter update'
    result=(run_dir/'selected.pt',dict(trainer.cv_timing),trainer.train_loader.dataset.parent_sampler.state_dict(),trainer.training_init_sha)
    del model,trainer,checkpoint; gc.collect(); torch.cuda.empty_cache()
    return result


def cv_frozen_identity(spec,checkpoint,valid):
    return dict(spec=spec,checkpoint_sha=file_sha(checkpoint),threshold_source='inner-valid',valid_stems=sorted(valid),evaluator=EVAL_VERSION)


def cv_require_frozen(path,spec,checkpoint,valid):
    assert Path(path).is_file(), 'Run freeze stage before evaluation'
    frozen=json.loads(Path(path).read_text())
    assert frozen['identity']==cv_frozen_identity(spec,checkpoint,valid), 'Frozen model/config/validation differs'
    assert set(map(int,frozen['thresholds']))=={0,1} and all(np.isfinite(v) and 0<=v<=1 for v in frozen['thresholds'].values())
    return frozen


def cv_freeze(run_dir,spec,recipe,valid):
    path=run_dir/'freeze.json'; cp=run_dir/'selected.pt'; trained=json.loads((run_dir/'training.json').read_text())
    assert trained['spec']==spec and trained['checkpoint_sha']==file_sha(cp)
    if path.is_file(): return cv_require_frozen(path,spec,cp,valid)
    model=load_model(recipe,cp); predictions=predict_model(model,recipe,valid); validate_predictions(predictions,valid,'segment')
    frozen=dict(identity=cv_frozen_identity(spec,cp,valid),thresholds=select_thresholds(predictions,valid),created_utc=datetime.now(timezone.utc).isoformat(),status='frozen before heldout prediction')
    cv_atomic_json(path,frozen)
    del predictions,model; gc.collect(); runtime()[0].cuda.empty_cache()
    return frozen


def cv_save_predictions(predictions,path,task):
    from pycocotools import mask as mask_utils
    with gzip.open(path,'wt',encoding='utf-8') as stream:
        for stem,items in predictions.items():
            packed=[]
            for cls,score,value in items:
                if task=='segment':
                    value=mask_utils.encode(np.asfortranarray(np.asarray(value,np.uint8)))
                    value=dict(size=value['size'],counts=value['counts'].decode('ascii'))
                else: value=np.asarray(value,float).tolist()
                packed.append([int(cls),float(score),value])
            stream.write(json.dumps(dict(stem=stem,predictions=packed),allow_nan=False)+'\n')


def cv_evaluate(run_dir,spec,recipe,valid,held,out,name,scope):
    cp=run_dir/'selected.pt'; freeze_path=run_dir/'freeze.json'
    frozen=cv_require_frozen(freeze_path,spec,cp,valid)
    model=load_model(recipe,cp); predictions=predict_model(model,recipe,held); validate_predictions(predictions,held,'segment')
    ops=operating_metrics(predictions,held,{int(k):v for k,v in frozen['thresholds'].items()}); ops.to_csv(out/(name+'_operating.csv'),index=False)
    prediction_path=out/(name+'_predictions.jsonl.gz'); cv_save_predictions(predictions,prediction_path,'segment')
    record=dict(status='evaluated',name=name,spec=spec,model=recipe['name'],fold=spec['job']['fold'],seed=spec['job']['seed'],arm=spec['job']['arm'],freeze_sha=file_sha(freeze_path),checkpoint_sha=file_sha(cp),threshold_source='inner-valid',thresholds=frozen['thresholds'],evaluation_stems=sorted(held),box=canonical_coco(predictions,held,'bbox'),mask=canonical_coco(predictions,held,'segm'),predictions=str(prediction_path),predictions_sha=file_sha(prediction_path),scope=scope)
    cv_atomic_json(out/(name+'.json'),record)
    del predictions,model; gc.collect(); runtime()[0].cuda.empty_cache()
    return record


def run_full_cv(plan,token):
    assert not any([DO_TRAIN,DO_CV_SMOKE,DO_EVAL_TEST,DO_ABLATIONS,DO_EXPORT])
    assert sum([DO_FULL_CV,DO_CV_FREEZE,DO_CV_EVALUATE])==1, 'Train, freeze and evaluation are separate invocations'
    assert CV_APPROVED_PROTOCOL==token, 'Owner must approve exact protocol ID'
    previews=(EXP/'results/nb16_v2').glob('*/cv_preview_'+token+'.json')
    assert any(file_sha(p)==CV_APPROVED_PREVIEW_SHA and json.loads(p.read_text())['plan']==plan for p in previews), 'Approve production preview SHA before running'
    smokes=list((EXP/'results/nb16_v2').glob('*/cv_pipeline_smoke_'+token+'.json'))
    assert any(json.loads(p.read_text()).get('status')=='PASS' and json.loads(p.read_text()).get('plan')==plan for p in smokes), 'Run bounded production-batch smoke before full runs'
    torch,device=seed_everything(42); assert torch.cuda.is_available(); ensure_coco()
    out=EXP/'results/nb16_recipe_v2'/token; runs=EXP/'runs/nb16_recipe_v2'/token
    out.mkdir(parents=True,exist_ok=True); runs.mkdir(parents=True,exist_ok=True); cv_atomic_json(out/'plan.json',plan)
    assignments=pd.read_csv(CV_ROOT/'assignments.csv',keep_default_na=False); records=[]; errors=[]
    for job in plan['jobs']:
        recipe,fold,seed,arm=job['recipe'],job['fold'],job['seed'],job['arm']
        name=f'{recipe["name"]}_fold{fold}_seed{seed}_{arm}'; run_dir=runs/name; root=CV_ROOT/f'fold{fold}'/arm
        parents=assignments[(assignments.fold==fold)&(assignments.role=='train')]; steps=math.ceil(len(parents)/recipe['batch'])
        spec=dict(plan_id=token,job=job,steps_per_epoch=steps,optimizer_steps=steps*recipe['epochs'],initializer_sha=file_sha(EXP/'weights'/recipe['init']),cv_id=CV_HANDOFF['cv_id'],dataset_id=DATA_ID,source_id=CODE_ID,train_parent_stems=sorted(parents.stem))
        try:
            run_dir.mkdir(parents=True,exist_ok=True); spec_path=run_dir/'spec.json'
            if spec_path.exists(): assert json.loads(spec_path.read_text())==spec
            else:
                assert not any(run_dir.iterdir()), 'Incomplete run without identity'
                cv_atomic_json(spec_path,spec)
            training_path=run_dir/'training.json'; valid=load_gt(root/'valid')
            assert set(valid)==set(assignments.loc[(assignments.fold==fold)&(assignments.role=='valid'),'stem'])
            if DO_FULL_CV:
                if training_path.is_file():
                    record=json.loads(training_path.read_text()); assert record['spec']==spec and record['checkpoint_sha']==file_sha(run_dir/'selected.pt')
                else:
                    cp,timing,state,init_sha=cv_train_yolo(recipe,seed,run_dir,root,valid,steps,spec)
                    curve=pd.read_csv(run_dir/'selection.csv'); assert curve.optimizer_steps.tolist()==[steps*(e+1) for e in range(recipe['epochs'])]
                    usage=CVParentSampler(cv_training_rows(fold,arm),seed,arm=='offline_aug'); usage.load_state_dict(state)
                    record=dict(status='trained; not evaluated',name=name,spec=spec,checkpoint=str(cp),checkpoint_sha=file_sha(cp),optimizer_steps=spec['optimizer_steps'],selected_epoch=int(curve.loc[curve.val_AP.idxmax(),'epoch']),training_init_sha=init_sha,timing=timing,sampling=usage.summary(),sampling_seed=seed,gpu=torch.cuda.get_device_name())
                    usage.table().to_csv(run_dir/'sampling_counts.csv',index=False); cv_atomic_json(training_path,record)
                counterpart=runs/f'{recipe["name"]}_fold{fold}_seed{seed}_{"control" if arm=="offline_aug" else "offline_aug"}'/'training.json'
                if counterpart.is_file():
                    other=json.loads(counterpart.read_text()); assert other['training_init_sha']==record['training_init_sha'] and other['optimizer_steps']==record['optimizer_steps'] and other['sampling']['output_images']==record['sampling']['output_images']
                records.append(record)
            elif DO_CV_FREEZE: records.append(cv_freeze(run_dir,spec,recipe,valid))
            else:
                cv_require_frozen(run_dir/'freeze.json',spec,run_dir/'selected.pt',valid)
                held=load_gt(root/'test'); assert set(held)==set(assignments.loc[(assignments.fold==fold)&(assignments.role=='heldout'),'stem']) and set(held).isdisjoint(valid)
                records.append(cv_evaluate(run_dir,spec,recipe,valid,held,out,name,plan['scope']))
        except Exception as error:
            import traceback
            failure=dict(name=name,error=type(error).__name__,message=str(error),traceback=traceback.format_exc()); errors.append(failure)
            cv_atomic_json(out/(name+'_error.json'),failure); print('FAILED',name,failure['message'])
        finally:
            gc.collect(); torch.cuda.empty_cache()
            cv_atomic_json(out/'progress.json',dict(stage='train' if DO_FULL_CV else 'freeze' if DO_CV_FREEZE else 'evaluate',completed=len(records),failed=len(errors),total=len(plan['jobs']),current=name,status='running'))
    cv_atomic_json(out/'progress.json',dict(completed=len(records),failed=len(errors),total=len(plan['jobs']),status='completed' if not errors and len(records)==len(plan['jobs']) else 'finished_with_errors'))
    if DO_CV_EVALUATE and records:
        table=pd.DataFrame([dict(model=r['model'],fold=r['fold'],seed=r['seed'],arm=r['arm'],mask_AP=r['mask']['AP'],box_AP=r['box']['AP']) for r in records]); table.to_csv(out/'results.csv',index=False)
        table.groupby(['model','arm'])[['mask_AP','box_AP']].agg(['mean','std','count']).to_csv(out/'fold_summary.csv')
    if errors: raise RuntimeError(f'{len(errors)} jobs failed; inspect progress/errors, do not report a complete comparison')
    return out

### 15.2 Preflight, external ZIP audit และ Stripe benchmark

ค่าเริ่มต้น **ปิดทุก action** `NB16_CV_PREVIEW=1` สร้าง preview จาก production loader; `NB16_CV_SMOKE=1` เปิด bounded production-batch smoke ของ n/s-seg × 2 arms, fold 0, seed 42 ไม่เกิน 2 optimizer steps ต่อ arm/model ไม่มี outer prediction บันทึก finite/nonzero gradients, weight changes, paired initializer/output budget และ GPU memory ไม่ใช่คะแนนวิจัย

ก่อน production ต้องมี v2 audited handoff, preview ที่เจ้าของตรวจ และ smoke ที่ตรง protocol จากนั้นตั้ง `NB16_APPROVED_PROTOCOL` และ `NB16_APPROVED_PREVIEW_SHA` ตามค่าที่ notebook พิมพ์ เปิด `NB16_FULL_CV=1` เฉพาะ training ต่อมาเปิด `NB16_CV_FREEZE=1` และ `NB16_CV_EVALUATE=1` คนละครั้ง งบ draft ปัจจุบัน n/s-seg × 2 arms × 5 folds × seed 42 = 20 runs; 30 virtual epochs, n batch12/s batch6 ยังต้องผ่าน smoke สูตรใหม่

`NB16_EXTERNAL_AUDIT=1` audit ZIP ทีละชุดโดยไม่ extract เข้า dataset ถาวร ตรวจ names/order, annotation geometry/raster, pairing, SHA/decoded duplicates และ dHash candidates เทียบ source pool และระหว่าง external sets Semantic annotation/ontology/duplicate adjudication เป็นงานผู้ตรวจจริง ผล mechanical audit ไม่ยืนยัน independence ให้กรอก `EXTERNAL_REVIEWS` ผูกกับ audit ID ก่อน `NB16_EVAL_EXTERNAL=1` รายงานแยกชุดเสมอและไม่เปลี่ยน thresholds จาก freeze

Stripe 88 เป็น **development-inspected, seen-scene test** ไม่ใช่ fixed test 105/CV outer-heldout ต้องตั้ง `STRIPE_TEST_ROOT` เป็น split directory ที่มี `images/labels` จริงและจำนวน 88 เท่านั้น ยังไม่เดา path เปิด `NB16_EVAL_STRIPE=1` พร้อม `EXTERNAL_RUN_DIR` ของ run ที่ freeze แล้วเท่านั้น ชุด external/Stripe ใช้ original bytes กับ evaluation preprocessing ปกติ ไม่มี augmentation การตรวจประวัติว่าคะแนน test เคยใช้ตัดสินใจอะไรยังเป็นข้อจำกัด ไม่สร้างหลักฐานขึ้นเอง

In [24]:
DO_EXTERNAL_AUDIT = os.environ.get('NB16_EXTERNAL_AUDIT','0')=='1'
DO_EXTERNAL_EVAL = os.environ.get('NB16_EVAL_EXTERNAL','0')=='1'
DO_STRIPE_EVAL = os.environ.get('NB16_EVAL_STRIPE','0')=='1'
EXTERNAL_ZIPS = {'external_1':EXP/'data/Piled Materials.v2i.yolo26.zip','external_2':EXP/'data/S02.zip','external_3':EXP/'data/Separate Rock-Sand.v6i.yolo26.zip'}
EXTERNAL_REVIEWS = {}
EXTERNAL_RUN_DIR = None
STRIPE_TEST_ROOT = None


def cv_sampler_checks():
    rows=pd.DataFrame([dict(parent_stem=f'parent{i}',variant=v,image=f'{i}_{v}.png',role='train') for i in range(3) for v in CV_VARIANTS])
    aug=CVParentSampler(rows,42,True); base=CVParentSampler(rows[rows.variant=='original'],42,False)
    for _ in range(100):
        aug.draw(True); base.draw(True); assert aug.draws[-1]['parent']==base.draws[-1]['parent']
        aug.draw(False)
    state=aug.state_dict(); replay=CVParentSampler(rows,42,True); replay.load_state_dict(state)
    assert [aug.draw(True) for _ in range(20)]==[replay.draw(True) for _ in range(20)]
    assert aug.summary()==replay.summary()
    try: CVParentSampler(rows.iloc[:-1],42,True)
    except AssertionError: pass
    else: raise AssertionError('Incomplete variant lineage accepted')
    print('PASS parent-uniform pairing, donor separation, sampler resume and incomplete-lineage rejection')


def cv_pipeline_smoke(plan,token):
    assert not any([DO_FULL_CV,DO_CV_FREEZE,DO_CV_EVALUATE,DO_TRAIN,DO_INFER,DO_EVAL_TEST,DO_EXPORT,DO_EXTERNAL_EVAL,DO_STRIPE_EVAL])
    torch,_=runtime(); assert torch.cuda.is_available(); ensure_coco()
    root=EXP/'runs/nb16_recipe_v2_smoke'/STAMP; root.mkdir(parents=True,exist_ok=False)
    valid_all=load_gt(CV_ROOT/'fold0/control/valid'); chosen=smoke_parents(CV_LINEAGE[(CV_LINEAGE.fold==0)&(CV_LINEAGE.arm=='control')],'valid')
    valid={k:valid_all[k] for k in chosen}; records=[]
    for model_recipe in CV_FULL_RECIPES:
        pair=[]
        for arm in CV_ARMS:
            recipe=dict(model_recipe,epochs=1); job=dict(recipe=recipe,fold=0,seed=42,arm=arm)
            spec=dict(plan_id=token,job=job,steps_per_epoch=2,optimizer_steps=2,initializer_sha=file_sha(EXP/'weights'/recipe['init']),cv_id=CV_HANDOFF['cv_id'],dataset_id=DATA_ID,source_id=CODE_ID,scope='bounded production-batch smoke; no outer predictions')
            run_dir=root/(recipe['name']+'_'+arm); run_dir.mkdir(); cv_atomic_json(run_dir/'spec.json',spec)
            torch.cuda.reset_peak_memory_stats()
            cp,timing,state,init_sha=cv_train_yolo(recipe,42,run_dir,CV_ROOT/'fold0'/arm,valid,2,spec)
            committed=cv_resume_load(run_dir/'resume.pt',spec)
            assert committed['cv_steps']==2 and committed['training_final_sha']!=init_sha
            sampler=CVParentSampler(cv_training_rows(0,arm),42,arm=='offline_aug'); sampler.load_state_dict(state)
            assert sampler.outputs==2*recipe['batch']
            record=dict(model=recipe['name'],arm=arm,optimizer_steps=2,output_images=sampler.outputs,sources=sampler.summary(),finite_nonzero_gradients=True,training_init_sha=init_sha,peak_allocated_gib=torch.cuda.max_memory_allocated()/2**30,checkpoint_sha=file_sha(cp),resume_sha=file_sha(run_dir/'resume.pt'),timing=timing,outer_predictions=0)
            pair.append(record); records.append(record); del committed; gc.collect(); torch.cuda.empty_cache()
        assert pair[0]['training_init_sha']==pair[1]['training_init_sha'] and pair[0]['output_images']==pair[1]['output_images']
    path=OUT/('cv_pipeline_smoke_'+token+'.json')
    cv_atomic_json(path,dict(status='PASS',plan=plan,records=records,outer_predictions=0,scope='preflight only; not recipe benefit'))
    print('PASS bounded production-batch smoke:',path)
    return path


def eval_image_record(name,raw,label):
    from shapely.geometry import Polygon
    image=cv2.imdecode(np.frombuffer(raw,np.uint8),cv2.IMREAD_COLOR)
    assert image is not None and image.size<=180000000, 'Invalid/oversized image'
    counts=[0,0]; h,w=image.shape[:2]
    for line in label.splitlines():
        if not line.strip(): continue
        fields=line.split(); assert len(fields)>=7 and (len(fields)-1)%2==0
        cls=int(fields[0]); xy=np.asarray(fields[1:],float).reshape(-1,2)
        assert cls in (0,1) and np.isfinite(xy).all() and ((xy>=0)&(xy<=1)).all()
        polygon=Polygon(xy); assert polygon.is_valid and polygon.area>0
        mask=np.zeros((h,w),np.uint8); cv2.fillPoly(mask,[(xy*[w,h]).astype(np.int32)],1); assert mask.any()
        counts[cls]+=1
    gray=cv2.resize(cv2.cvtColor(image,cv2.COLOR_BGR2GRAY),(9,8),interpolation=cv2.INTER_AREA)
    return dict(image=name,image_sha=hashlib.sha256(raw).hexdigest(),label_sha=hashlib.sha256(label.encode()).hexdigest(),decoded_sha=hashlib.sha256(image.tobytes()+str(image.shape).encode()).hexdigest(),dhash=int.from_bytes(np.packbits((gray[:,1:]>gray[:,:-1]).ravel()).tobytes(),'big'),gravel=counts[0],sand=counts[1],height=h,width=w)


def external_reference_records():
    records=[]
    for row in MANIFEST.itertuples(index=False):
        label=Path(row.path).parent.parent/'labels'/(row.stem+'.txt')
        record=eval_image_record(row.image,Path(row.path).read_bytes(),label.read_text())
        record.update(dataset='source_pool',split=row.split); records.append(record)
    return records


def audit_external_zip(name,path,references):
    import zipfile,yaml
    from pathlib import PurePosixPath
    path=Path(path); rows=[]; failures=[]; matches=[]
    with zipfile.ZipFile(path) as archive:
        infos=archive.infolist(); assert len({i.filename for i in infos})==len(infos), 'Duplicate ZIP members'
        assert sum(i.file_size for i in infos)<=2*2**30, 'ZIP too large for bounded audit'
        for info in infos:
            member=PurePosixPath(info.filename)
            assert not member.is_absolute() and '..' not in member.parts and '\\' not in info.filename
            assert info.file_size<=64*2**20 and (info.external_attr>>16)&0o170000!=0o120000
        members={i.filename for i in infos if not i.is_dir()}
        configs=[m for m in members if PurePosixPath(m).name=='data.yaml']; assert len(configs)==1, 'Ambiguous/missing ontology config'
        metadata=yaml.safe_load(archive.read(configs[0])); names=metadata['names']
        if isinstance(names,dict):
            assert sorted(map(int,names))==[0,1]
            names=[names[k] for k in sorted(names,key=int)]
        ontology_matches=names==CLS
        images=sorted(m for m in members if PurePosixPath(m).suffix.lower() in {'.jpg','.jpeg','.png','.bmp'} and PurePosixPath(m).parent.name=='images')
        expected_labels={str(PurePosixPath(m).parent.parent/'labels'/(PurePosixPath(m).stem+'.txt')) for m in images}
        actual_labels={m for m in members if PurePosixPath(m).parent.name=='labels' and m.endswith('.txt')}
        assert images and actual_labels==expected_labels, 'ZIP image/label mismatch'
        for image_name in images:
            label_name=str(PurePosixPath(image_name).parent.parent/'labels'/(PurePosixPath(image_name).stem+'.txt'))
            try:
                row=eval_image_record(image_name,archive.read(image_name),archive.read(label_name).decode('utf-8'))
                row['label_member']=label_name; rows.append(row)
            except (AssertionError,ValueError,UnicodeError) as error:
                failures.append(dict(image=image_name,error=str(error)))
    for row in rows:
        for ref in references+rows:
            if ref is row: continue
            distance=(row['dhash']^ref['dhash']).bit_count()
            exact=row['decoded_sha']==ref['decoded_sha'] or row['image_sha']==ref['image_sha']
            if exact or distance<=6: matches.append(dict(image=row['image'],reference=ref['image'],reference_dataset=ref.get('dataset',name),exact=exact,hamming=distance))
    identity=dict(name=name,zip_sha=file_sha(path),source_dataset_id=DATA_ID,source_id=CODE_ID,names=names,images=rows)
    return dict(audit_id=cv_json_id(identity),identity=identity,mechanical_pass=ontology_matches and not failures and len(rows)==len(images),ontology_matches=ontology_matches,annotation_failures=failures,duplicate_candidates=matches,ontology_review='PENDING',annotation_semantics_review='PENDING',duplicate_review='PENDING',independent_holdout=False)


def require_external_review(audit,review):
    assert audit['mechanical_pass'], 'External mechanical audit failed'
    assert review.get('audit_id')==audit['audit_id']
    assert all(review.get(k) is True for k in ['ontology_approved','annotations_approved','duplicate_review_approved'])
    assert review.get('reviewer') and review.get('evidence'), 'Record reviewer and adjudication evidence'
    return 'external audited benchmark; independent acquisition not established'


def external_run_context(run_dir):
    run_dir=Path(run_dir).resolve()
    assert run_dir.is_relative_to((EXP/'runs/nb16_recipe_v2').resolve())
    spec=json.loads((run_dir/'spec.json').read_text())
    assert spec['source_id']==CODE_ID and spec['dataset_id']==DATA_ID and spec['cv_id']==CV_HANDOFF['cv_id'] and spec['plan_id']==CV_PROTOCOL_ID
    recipe=spec['job']['recipe']; root=CV_ROOT/f'fold{spec["job"]["fold"]}'/spec['job']['arm']; valid=load_gt(root/'valid')
    cv_require_frozen(run_dir/'freeze.json',spec,run_dir/'selected.pt',valid)
    return run_dir,spec,recipe,valid


def evaluate_external_zip(name,path,audit,review,run_dir):
    import zipfile,tempfile
    scope=require_external_review(audit,review)
    assert file_sha(path)==audit['identity']['zip_sha']
    run_dir,spec,recipe,valid=external_run_context(run_dir)
    out=EXP/'results/nb16_recipe_v2'/spec['plan_id']/'external'/run_dir.name/name/audit['audit_id'][:20]
    out.mkdir(parents=True,exist_ok=True)
    with tempfile.TemporaryDirectory(prefix='aggregate-external-eval-') as temporary:
        root=Path(temporary); (root/'images').mkdir(); (root/'labels').mkdir()
        with zipfile.ZipFile(path) as archive:
            for row in audit['identity']['images']:
                stem=hashlib.sha256(row['image'].encode()).hexdigest()[:24]
                target=root/'images'/(stem+Path(row['image']).suffix.lower()); label=root/'labels'/(stem+'.txt')
                target.write_bytes(archive.read(row['image'])); label.write_bytes(archive.read(row['label_member']))
                assert file_sha(target)==row['image_sha'] and hashlib.sha256(label.read_text().encode()).hexdigest()==row['label_sha']
        record=cv_evaluate(run_dir,spec,recipe,valid,load_gt(root),out,name,scope)
    record.update(external_audit=audit,review=review,independent_holdout=False)
    cv_atomic_json(out/(name+'.json'),record)
    return record


def evaluate_stripe(run_dir,test_root):
    run_dir,spec,recipe,valid=external_run_context(run_dir); test_root=Path(test_root).resolve()
    images=sorted(p for p in (test_root/'images').iterdir() if p.suffix.lower() in {'.jpg','.jpeg','.png','.bmp'})
    assert len(images)==88 and len({p.stem for p in images})==88, 'Stripe must be the explicitly supplied 88-image benchmark'
    assert {p.stem for p in images}=={p.stem for p in (test_root/'labels').glob('*.txt')}
    assert not any('__cvaug_' in p.stem or p.stem.endswith(('_aug1','_aug2','_aug3')) for p in images)
    rows=[eval_image_record(p.name,p.read_bytes(),(test_root/'labels'/(p.stem+'.txt')).read_text()) for p in images]
    dataset_id=cv_json_id(rows); out=EXP/'results/nb16_recipe_v2'/spec['plan_id']/'stripe'/run_dir.name/dataset_id[:20]; out.mkdir(parents=True,exist_ok=True)
    record=cv_evaluate(run_dir,spec,recipe,valid,load_gt(test_root),out,'stripe_88','development-inspected, seen-scene test; not untouched')
    record.update(dataset_id=dataset_id,source_root=str(test_root),test_decision_history_audit='PENDING; no claim that scores were never used for development')
    cv_atomic_json(out/'stripe_88.json',record)
    return record


cv_sampler_checks()
if CV_HANDOFF is None:
    assert not any([DO_FULL_CV,DO_CV_PREVIEW,DO_CV_SMOKE,DO_CV_FREEZE,DO_CV_EVALUATE,DO_EXTERNAL_EVAL,DO_STRIPE_EVAL]), 'Missing v2 handoff; do not use v1'
    print('V2 pending 00 audit/build. No dataset creation/training/evaluation executed.')
else:
    CV_PLAN,CV_PROTOCOL_ID=cv_plan()
    print('DRAFT protocol ID:',CV_PROTOCOL_ID,'| jobs:',len(CV_PLAN['jobs']))
    assert sum([DO_FULL_CV,DO_CV_PREVIEW,DO_CV_SMOKE,DO_CV_FREEZE,DO_CV_EVALUATE,DO_EXTERNAL_EVAL,DO_STRIPE_EVAL])<=1, 'One stage per invocation'
    if DO_CV_PREVIEW: CV_PREVIEW_PATH=cv_preview(CV_PLAN,CV_PROTOCOL_ID)
    if DO_CV_SMOKE: CV_SMOKE_PATH=cv_pipeline_smoke(CV_PLAN,CV_PROTOCOL_ID)
    if any([DO_FULL_CV,DO_CV_FREEZE,DO_CV_EVALUATE]): FULL_CV_OUT=run_full_cv(CV_PLAN,CV_PROTOCOL_ID)
if DO_EXTERNAL_AUDIT or DO_EXTERNAL_EVAL:
    references=external_reference_records(); audits={}
    for name,path in EXTERNAL_ZIPS.items():
        audit=audit_external_zip(name,path,references); audits[name]=audit; cv_atomic_json(OUT/(name+'_audit.json'),audit)
        references.extend(dict(row,dataset=name) for row in audit['identity']['images'])
    for name,audit in audits.items():
        if DO_EXTERNAL_EVAL:
            assert EXTERNAL_RUN_DIR is not None
            evaluate_external_zip(name,EXTERNAL_ZIPS[name],audit,EXTERNAL_REVIEWS.get(name,{}),EXTERNAL_RUN_DIR)
if DO_STRIPE_EVAL:
    assert EXTERNAL_RUN_DIR is not None and STRIPE_TEST_ROOT is not None
    evaluate_stripe(EXTERNAL_RUN_DIR,STRIPE_TEST_ROOT)

PASS parent-uniform pairing, donor separation, sampler resume and incomplete-lineage rejection
DRAFT protocol ID: 3b867e9545011a5f3c52 | jobs: 20


In [25]:
DO_BATCH_PROBE = os.environ.get('NB16_CV_PROBE','0') == '1'
CV_PROBE_STEPS = 3


def run_batch_probe():
    import shutil
    assert CV_HANDOFF is not None and len(CV_PREFLIGHT)==30
    assert not DO_FULL_CV and not DO_TRAIN and not DO_CV_SMOKE and not DO_EVAL_TEST and not DO_ABLATIONS, 'Use batch-probe opt-in alone'
    ensure_coco(); torch,device = seed_everything(42)
    assert torch.cuda.is_available(), 'Batch probe requires GPU'
    assignments = pd.read_csv(CV_ROOT/'assignments.csv',keep_default_na=False)
    root = CV_ROOT/'fold0'/'control'
    valid = load_gt(root/'valid')
    expected = set(assignments.loc[(assignments.fold==0)&(assignments.role=='valid'),'stem'])
    assert set(valid)==expected
    out = EXP/'results/nb16_batch_probe'/STAMP
    runs = EXP/'runs/nb16_batch_probe'/STAMP
    out.mkdir(parents=True,exist_ok=False); runs.mkdir(parents=True,exist_ok=False)
    rows=[]
    for recipe in CV_FULL_RECIPES:
        probe_recipe = dict(recipe,epochs=1)
        run_dir = runs/recipe['name']; run_dir.mkdir(parents=True,exist_ok=False)
        torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
        started = time.perf_counter()
        try:
            if recipe['kind']=='yolo': cv_train_yolo(probe_recipe,42,run_dir,root,valid,CV_PROBE_STEPS)
            else: cv_train_torch(probe_recipe,42,run_dir,root,valid,CV_PROBE_STEPS)
            status='PASS'; error=None
        except Exception as exc:
            status='FAIL'; error=f'{type(exc).__name__}: {exc}'
            print('BATCH PROBE FAILED',recipe['name'],error,flush=True)
        rows.append(dict(model=recipe['name'],batch=recipe['batch'],status=status,peak_allocated_gib=float(torch.cuda.max_memory_allocated()/2**30),peak_reserved_gib=float(torch.cuda.max_memory_reserved()/2**30),seconds=time.perf_counter()-started,error=error))
        write_json(out/(recipe['name']+'.json'),rows[-1])
        gc.collect(); torch.cuda.empty_cache()
    table = pd.DataFrame(rows)
    table.to_csv(out/'batch_probe.csv',index=False)
    write_json(out/'batch_probe.json',dict(recipes=CV_FULL_RECIPES,steps=CV_PROBE_STEPS,results=rows))
    display(table)
    if table.status.ne('PASS').any(): raise RuntimeError('At least one batch probe failed; lower the failed recipe batch before full CV.')
    return table


if DO_BATCH_PROBE:
    BATCH_PROBE_TABLE = run_batch_probe()
else:
    print('Batch probe disabled. Set NB16_CV_PROBE=1 to validate per-recipe batches in notebook before full CV.')

Batch probe disabled. Set NB16_CV_PROBE=1 to validate per-recipe batches in notebook before full CV.
